# SAM 3.1: Image Segmentation and Time-lapse Tracking

Segment **grayscale or RGB TIFF, JPG, or PNG images** with text, foreground/background points,
or rectangles. Track multiple objects forward and backward from any selected frame,
correct their masks, and export segmentation images at the original image size.

**Before you start:** select a Python 3.12 GPU runtime with native BF16 support (A100 preferred;
T4 is not supported here). Request access to [SAM 3.1 weights](https://huggingface.co/facebook/sam3.1).
Text-prompt results depend on image content; point prompts also work without text.
Review masks and object identities before exporting.

Your original images remain unchanged. The model sees 8-bit RGB PNG images. The exported
results contain only segmentation images and a JSON file with prompts and reproduction settings.
This notebook processes **2D images and 2D time-lapse sequences**, with uint8 or uint16
grayscale or RGB TIFF pixels, 8-bit grayscale/RGB JPG/JPEG or PNG, or 16-bit grayscale PNG.
Non-RGB multichannel images and Z stacks are not supported.

**Validation status:** CPU image/session tests accompany the source. GPU inference,
Colab widget behavior, and accuracy on your images must be checked in your runtime.
Start with a short sequence and two objects before processing a longer recording.

## 1. Install the required packages

Use **Runtime > Change runtime type > Runtime Version** to select a version with Python
3.12 (for example, 2026.07 when available), then choose a compatible GPU. Python 3.13
is incompatible with this notebook's NumPy 1.26.4 requirement.

Run this cell once in a fresh GPU runtime. It installs a fixed SAM source revision and
the notebook dependencies without replacing PyTorch. After installation and a fresh-process
import check succeed, the session restarts automatically to clear previously loaded libraries.
A temporary disconnection or unexpected-session-exit notification is expected. Reconnect if
needed, then continue at **section 2**; do not run section 1 again unless reinstalling.
Variables and loaded models are cleared on restart; save your project before reinstalling
during an analysis. Failed installation or import checks stop without restarting.
Model weights are downloaded later.

In [ ]:
import subprocess
import sys

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Use a Python 3.12 runtime: Runtime > Change runtime type > Runtime Version. Then run section 1 and restart the session.")
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_capability()[0] < 8:
    raise RuntimeError("Select an A100 or another GPU with native BF16 support. T4 is not supported.")
SAM_REVISION = "7f473cfd983d5eb43040ff608f66bff0d7bdcb53"
packages = [
    "setuptools<81", "numpy==1.26.4", "tifffile>=2024.8.30",
    "imagecodecs", "Pillow>=10", "ipywidgets>=8,<9", "ipympl>=0.9,<0.10",
    f"sam3[notebooks] @ git+https://github.com/facebookresearch/sam3.git@{SAM_REVISION}",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=numpy", *packages])
subprocess.check_call([sys.executable, "-c",
    "import numpy; import numpy.random; import sklearn; import skimage; import matplotlib; import torchvision; "
    "assert numpy.__version__ == '1.26.4', numpy.__version__; "
    "print('NumPy and image-library imports passed in a fresh Python process.')"])
print("Installation and import checks passed. Restarting the session now; reconnect and run section 2 onward.", flush=True)
try:
    import google.colab
except ImportError:
    print("Outside Colab: restart your Python kernel manually, then run section 2 onward.", flush=True)
else:
    import os
    import signal
    os.kill(os.getpid(), signal.SIGKILL)

## 2. Load the notebook tools

Run the following cell once after restarting. The tools are included in this notebook;
you do not need to upload other Python files or publish a repository.

In [ ]:
#@title Load the included tools
import sys
from pathlib import Path
tool_root = Path("/content/sam31_notebook") if Path("/content").is_dir() else Path.cwd() / ".sam31_runtime"
package_dir = tool_root / "sam31_colab"
package_dir.mkdir(parents=True, exist_ok=True)

helper_sources = {'__init__.py': '"""SAM 3.1 Colab support modules."""\n\nfrom .io import ImageSettings, TiffSequence, labels_from_masks\n\n__all__ = ["ImageSettings", "TiffSequence", "labels_from_masks"]\n', 'colab_ui.py': '"""Display annotation controls through Colab HTML and Python callbacks."""\n\nimport base64\nfrom io import BytesIO\nimport json\nimport math\nfrom uuid import uuid4\n\nimport numpy as np\nfrom PIL import Image\n\nfrom .labels import draw_object_label\n\n\nclass ColabTrackingUI:\n    """Use show() in Colab; close() leaves the tracking project and its prompts intact."""\n\n    def __init__(self, project, initial_pending=None):\n        self.project = project\n        self.frame = 0\n        self.object_id = None\n        self.low, self.high = float(project.prepared["low"]), float(project.prepared["high"])\n        self.alpha = 0.4\n        saved = project.concept_prompt\n        self.concept = saved["text"] if saved is not None else ""\n        self.box_update = "add"\n        self.name = ""\n        self.name_prefix = project.name_prefix\n        self.status = "Choose a frame, then enter a concept or add points."\n        self._closed = False\n        self._busy = False\n        self._callback_name = None\n        self._output = None\n        self.initial_pending = list(initial_pending or [])\n        self._root_id = "sam31_" + uuid4().hex\n\n    def _masks(self):\n        if self.project.rebuild_required:\n            return {}\n        if self.project.current_frame_index == self.frame:\n            return self.project.current_frame_outputs or {}\n        return {} if self.project.dirty else self.project.results.get(self.frame, {})\n\n    def state(self):\n        source_index = self.project.sequence.frame_indices[self.frame]\n        pixels = self.project.sequence.preview(source_index, low=self.low, high=self.high).copy()\n        height, width = self.project.sequence.shape[1:]\n        owner = np.zeros((height, width), np.int64)\n        shown = {}\n        for object_id, mask in sorted(self._masks().items()):\n            item = self.project.objects.get(object_id)\n            if item is None or item["excluded"]:\n                continue\n            mask = np.asarray(mask, dtype=bool)\n            if mask.shape != owner.shape:\n                raise ValueError("Mask dimensions do not match the image")\n            visible = mask & (owner == 0)\n            owner[visible] = object_id\n            pixels[visible] = np.rint((1 - self.alpha) * pixels[visible] + self.alpha * np.asarray(item["color"])).astype(np.uint8)\n            shown[object_id] = mask\n        image = Image.fromarray(pixels)\n        for object_id, mask in shown.items():\n            draw_object_label(image, mask, object_id, self.project.objects[object_id],\n                              selected=object_id == self.object_id)\n        buffer = BytesIO()\n        image.save(buffer, format="PNG")\n        active = [{"id": key, "name": item["name"], "color": item["color"]}\n                  for key, item in sorted(self.project.objects.items()) if not item["excluded"]]\n        if self.object_id is not None and self.object_id not in [item["id"] for item in active]:\n            self.object_id = None\n        if self.object_id is not None:\n            self.name = self.project.objects[self.object_id]["name"]\n        return {"frame": self.frame, "source_index": source_index,\n                "frame_count": self.project.sequence.shape[0], "width": width, "height": height,\n                "object_id": self.object_id, "objects": active, "name": self.name,\n                "concept": self.concept, "name_prefix": self.name_prefix,\n                "effective_name_prefix": self.project.effective_name_prefix,\n                "box_update": self.box_update, "saved_examples": self.project.concept_prompt,\n                "low": self.low, "high": self.high, "alpha": self.alpha,\n                "maximum": int(np.iinfo(self.project.sequence.metadata["dtype"]).max),\n                "image": "data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode("ascii"),\n                "tracked_frames": len(self.project.results),\n                "dirty": self.project.dirty, "rebuild_required": self.project.rebuild_required,\n                "closed": self._closed, "status": self.status}\n\n    def handle_request(self, action, payload=None):\n        if self._closed:\n            raise RuntimeError("This panel is closed. Use the latest displayed panel.")\n        if self._busy:\n            raise RuntimeError("Wait for the current operation to finish.")\n        payload = payload or {}\n        self._busy = True\n        try:\n            frame = payload.get("frame", self.frame)\n            if isinstance(frame, bool) or not isinstance(frame, int) or not 0 <= frame < self.project.sequence.shape[0]:\n                raise ValueError("Frame is outside the selected sequence")\n            selected = payload.get("object_id", self.object_id)\n            if selected is not None and (isinstance(selected, bool) or not isinstance(selected, int)\n                                         or selected not in self.project.objects or self.project.objects[selected]["excluded"]):\n                raise ValueError("Select an active object or New object")\n            low, high, alpha = [float(payload.get(key, getattr(self, key))) for key in ("low", "high", "alpha")]\n            maximum = np.iinfo(self.project.sequence.metadata["dtype"]).max\n            if not all(math.isfinite(value) for value in (low, high, alpha)) or not 0 <= low <= high <= maximum or not 0 <= alpha <= 1:\n                raise ValueError("Check display limits and opacity")\n            if action not in {"frame", "object", "display", "concept", "points", "name", "prefix", "exclude", "track"}:\n                raise ValueError("Unknown panel operation")\n            if action not in {"frame", "object", "display"} and self.project.rebuild_required:\n                raise RuntimeError("Rebuild the project before adding prompts or tracking")\n            prefix = payload.get("name_prefix", self.name_prefix)\n            if not isinstance(prefix, str):\n                raise ValueError("Name prefix must be a string")\n            box_update = payload.get("box_update", self.box_update)\n            if box_update not in ("add", "replace"):\n                raise ValueError("Box update must be add or replace")\n            name_changed = payload.get("name_changed", payload.get("name", self.name) != self.name)\n            self.frame, self.object_id = frame, selected\n            self.low, self.high, self.alpha = low, high, alpha\n            self.name_prefix = prefix.strip()\n            self.box_update = box_update\n            if "concept" in payload:\n                self.concept = str(payload["concept"])\n            if "name" in payload:\n                self.name = str(payload["name"])\n            pending = payload.get("pending", [])\n            if action == "concept":\n                if any(item["kind"] != "box" for item in pending):\n                    raise ValueError("Apply or clear pending points first")\n                self.project.update_concept(frame, self.concept,\n                                            boxes=[item["coordinates"] for item in pending],\n                                            box_labels=[item["label"] for item in pending],\n                                            mode=self.box_update, name_prefix=self.name_prefix)\n                self.object_id, self.name = None, ""\n                self.status = "Detection applied. Review the new objects and Track before exporting."\n            elif action == "points":\n                if any(item["kind"] != "point" for item in pending):\n                    raise ValueError("Apply or clear pending boxes first")\n                self._save_prefix()\n                previous_ids = set(self.project.objects)\n                returned_masks = self.project.add_points(frame, selected, [item["coordinates"] for item in pending],\n                                                         [item["label"] for item in pending],\n                                                         name=self.name if selected is None or name_changed else "")\n                added = sorted(set(self.project.objects) - previous_ids)\n                if selected is None and added:\n                    self.object_id = added[0]\n                if self.object_id not in returned_masks or not np.any(returned_masks[self.object_id]):\n                    self.status = "Points applied, but no visible mask was returned for this object. Check prompt coordinates and inference before tracking."\n                else:\n                    self.status = "Points applied. Review the masks."\n            elif action == "name":\n                if selected is None:\n                    raise ValueError("Select an existing object first")\n                self.project.rename_object(selected, self.name)\n                self.status = "Object name saved."\n            elif action == "prefix":\n                self.project.set_name_prefix(self.name_prefix, rename_existing=True)\n                self.status = "Prefix applied to all active objects and future objects."\n            elif action == "exclude":\n                if selected is None:\n                    raise ValueError("Select an existing object first")\n                self.project.remove_object(selected, frame=frame)\n                self.object_id, self.name = None, ""\n                self.status = "Object excluded. Track again to update results."\n            elif action == "track":\n                if pending:\n                    raise ValueError("Apply or clear pending annotations before tracking")\n                self._save_prefix()\n                self.project.propagate(start_frame=frame, direction=payload.get("direction", "both"),\n                                       progress=lambda completed, total: None)\n                self.status = ("Partial coverage. Track both directions before exporting." if self.project.dirty\n                               else "Tracking finished. All selected frames are ready for export.")\n            elif action in {"frame", "object"}:\n                if action == "object" and selected is None:\n                    self.name = ""\n                self.status = "View updated. Pending annotations were cleared."\n            else:\n                self.status = "Display updated; model input settings are unchanged."\n            return {"ok": True, "state": self.state()}\n        except Exception as error:\n            self.status = f"{type(error).__name__}: {error}"\n            return {"ok": False, "state": self.state()}\n        finally:\n            self._busy = False\n\n    def _save_prefix(self):\n        if self.name_prefix != self.project.name_prefix:\n            self.project.set_name_prefix(self.name_prefix)\n\n    def html(self):\n        return \'\'\'<div id="ROOT" style="font:14px Arial,sans-serif;color:#202124;background:#fff;padding:14px;border:1px solid #ccc;max-width:1100px">\n<style>#ROOT label{display:inline-flex;align-items:center;gap:6px;margin:5px}#ROOT button,#ROOT select,#ROOT input{font:inherit;padding:5px}#ROOT button{margin:4px;cursor:pointer}#ROOT canvas{display:block;touch-action:none;cursor:crosshair}#ROOT .row{display:flex;flex-wrap:wrap;align-items:center;gap:6px}#ROOT .viewport{max-height:650px;overflow:auto;border:1px solid #aaa;background:#222}#ROOT .status{white-space:pre-wrap;padding:8px;border:1px solid #ddd;margin-top:8px}#ROOT button:disabled{cursor:wait;opacity:.5}</style>\n<b>SAM 3.1 · Annotation and tracking</b>\n<p>Frames are zero-based. Click for points; drag for boxes. Zoom with the scale control and scroll to move around.</p>\n<div class="row"><label>Frame <input data-role="frame" type="range" min="0" value="0"></label><label>Frame number <input data-role="frame_number" type="number" min="0" value="0" style="width:80px"></label><span data-role="source"></span></div>\n<div class="row"><label>Object <select data-role="object"></select></label><label>Tool <select data-role="tool"><option value="point_positive">+ Point (target)</option><option value="point_negative">− Point (background)</option><option value="box_positive">+ Box</option><option value="box_negative">− Box</option></select></label><button data-action="undo">Undo pending</button><button data-action="clear">Clear pending</button><span data-role="pending"></span></div>\n<div class="row"><label>Concept <input data-role="concept" type="text" placeholder="e.g. dog"></label><label>Box update <select data-role="box_update" data-mutation><option value="add">Add to saved</option><option value="replace">Replace saved</option></select></label><button data-action="concept" data-mutation>Apply concept / boxes</button><button data-action="points" data-mutation>Apply points</button></div>\n<div class="row"><span data-role="saved"></span><span>Saved boxes are dashed. Reapplying detection replaces objects/IDs, discards point corrections and previous tracking; redo Track before export.</span></div>\n<div class="row"><label>Name prefix <input data-role="name_prefix" data-mutation type="text" placeholder="Blank: concept or Object"></label><button data-action="prefix" data-mutation>Apply prefix to all</button><span>Names use prefix + object ID. Applying to all replaces individual names.</span></div>\n<div class="row"><label>Name <input data-role="name" data-mutation type="text" placeholder="Optional individual name"></label><button data-action="name" data-mutation>Save name</button><button data-action="exclude" data-mutation>Exclude object</button></div>\n<div class="row"><label>Display low <input data-role="low" type="number" min="0" style="width:95px"></label><label>Display high <input data-role="high" type="number" min="0" style="width:95px"></label><label>Mask opacity <input data-role="alpha" type="range" min="0" max="1" step="0.05"></label><button data-action="display">Update display</button><label>Zoom <input data-role="zoom" type="range" min="0.1" max="4" step="0.1" value="1"></label><span data-role="zoom_value">100%</span></div>\n<div class="viewport"><canvas data-role="canvas"></canvas></div>\n<div data-role="legend"></div>\n<div class="row"><label>Direction <select data-role="direction"><option value="both">both</option><option value="forward">forward</option><option value="backward">backward</option></select></label><button data-action="track" data-mutation>Track</button><label>Tracked <progress data-role="progress" value="0" max="1"></progress></label><span data-role="coverage"></span></div>\n<div class="status" data-role="status" aria-live="polite"></div>\n</div>\'\'\'.replace("ROOT", self._root_id)\n\n    def script(self, state=None):\n        initial = {"state": state or self.state(), "pending": self.initial_pending,\n                   "root": self._root_id, "callback": self._callback_name}\n        encoded = json.dumps(initial, ensure_ascii=True).replace("<", "\\\\u003c").replace(">", "\\\\u003e").replace("&", "\\\\u0026")\n        return "(() => { const initial = " + encoded + ";\\n" + _SCRIPT + "\\n})()"\n\n    def _callback(self, action, payload=None):\n        from IPython.display import JSON\n        return JSON(self.handle_request(action, payload))\n\n    def show(self):\n        if self._closed:\n            raise RuntimeError("This panel is closed")\n        from google.colab import output\n        from IPython.display import HTML, display\n        if self._callback_name is not None:\n            self._output.unregister_callback(self._callback_name)\n        self._output = output\n        self._root_id = "sam31_" + uuid4().hex\n        self._callback_name = "sam31.panel." + uuid4().hex\n        output.register_callback(self._callback_name, self._callback)\n        display(HTML(self.html()))\n        output.eval_js(self.script(), timeout_sec=30)\n        self.initial_pending = []\n\n    def refresh(self):\n        """Display a fresh panel in the current cell; use the most recently displayed panel."""\n        self.name_prefix = self.project.name_prefix\n        saved = self.project.concept_prompt\n        if saved is not None:\n            self.concept = saved["text"]\n        self.show()\n\n    def close(self):\n        if not self._closed and self._callback_name is not None:\n            self._output.unregister_callback(self._callback_name)\n        self._closed = True\n\n\n_SCRIPT = r\'\'\'\nconst root = document.getElementById(initial.root);\nif (!root) throw new Error(\'Annotation panel HTML is missing. Run this cell again.\');\nconst get = role => root.querySelector(\'[data-role="\' + role + \'"]\');\nconst canvas = get(\'canvas\'), context = canvas.getContext(\'2d\');\nlet state = initial.state, pending = initial.pending || [], busy = false, drag = null, image = null;\nif(pending.length)get(\'tool\').value=pending[0].kind+\'_\'+(pending[0].label?\'positive\':\'negative\');\nconst allControls = [...root.querySelectorAll(\'input,button,select\')];\nfunction lock(value) {\n  busy = value;\n  allControls.forEach(control => control.disabled = value || state.closed || (state.rebuild_required && control.hasAttribute(\'data-mutation\')));\n  [\'name\',\'exclude\'].forEach(action => {root.querySelector(\'[data-action="\'+action+\'"]\').disabled = value || state.closed || state.rebuild_required || state.object_id === null;});\n}\nfunction point(event) {\n  const rect = canvas.getBoundingClientRect();\n  return [(event.clientX-rect.left)*canvas.width/rect.width, (event.clientY-rect.top)*canvas.height/rect.height];\n}\nfunction inside(position) {return position.every(Number.isFinite) && position[0]>=0 && position[0]<state.width && position[1]>=0 && position[1]<state.height;}\nfunction zoom() {\n  const scale = Number(get(\'zoom\').value);\n  canvas.style.width = state.width*scale+\'px\'; canvas.style.height = state.height*scale+\'px\';\n  get(\'zoom_value\').textContent = Math.round(scale*100)+\'%\';\n}\nfunction draw() {\n  if (!image) return;\n  context.clearRect(0,0,canvas.width,canvas.height); context.drawImage(image,0,0);\n  const scale = Number(get(\'zoom\').value);\n  context.lineWidth = 2/scale;\n  const saved=state.saved_examples;\n  if(saved && saved.frame===state.frame && saved.boxes.length){\n    context.save();context.setLineDash([6/scale,4/scale]);\n    saved.boxes.forEach((box,index)=>{\n      context.strokeStyle=saved.box_labels[index]?\'#39e68c\':\'#ff5e72\';\n      const [x,y,w,h]=box;context.strokeRect(x*state.width,y*state.height,w*state.width,h*state.height);\n    });\n    context.restore();\n  }\n  pending.forEach(item => {\n    context.strokeStyle = item.label ? \'#39e68c\' : \'#ff5e72\';\n    if (item.kind===\'point\') {\n      const [x,y]=item.coordinates, radius=6/scale;\n      context.beginPath(); context.moveTo(x-radius,y); context.lineTo(x+radius,y);\n      if(item.label){context.moveTo(x,y-radius);context.lineTo(x,y+radius);}\n      else {context.moveTo(x-radius,y-radius);context.lineTo(x+radius,y+radius);}\n      context.stroke();\n    } else {\n      const [x,y,w,h]=item.coordinates;\n      context.strokeRect(x*state.width,y*state.height,w*state.width,h*state.height);\n    }\n  });\n  if(drag && drag.end){context.strokeStyle=\'#ffd54f\';context.strokeRect(drag.start[0],drag.start[1],drag.end[0]-drag.start[0],drag.end[1]-drag.start[1]);}\n  get(\'pending\').textContent = pending.filter(item=>item.kind===\'point\').length+\' points / \'+pending.filter(item=>item.kind===\'box\').length+\' boxes pending\';\n}\nasync function render(next) {\n  state = next;\n  get(\'frame\').max=get(\'frame_number\').max=state.frame_count-1;\n  get(\'frame\').value=get(\'frame_number\').value=state.frame;\n  get(\'source\').textContent=\'Source frame \'+state.source_index+\' / local \'+state.frame;\n  const object=get(\'object\'); object.replaceChildren(new Option(\'New object\',\'\'));\n  state.objects.forEach(item=>object.add(new Option(item.id+\': \'+item.name,String(item.id))));\n  object.value=state.object_id===null?\'\':String(state.object_id);\n  get(\'name\').value=state.name; get(\'concept\').value=state.concept; get(\'name_prefix\').value=state.name_prefix;\n  get(\'box_update\').value=state.box_update;\n  const saved=state.saved_examples;\n  get(\'saved\').textContent=saved?(\'Saved examples: \'+saved.boxes.length+\' (\'+saved.box_labels.filter(label=>label===1).length+\' target / \'+saved.box_labels.filter(label=>label===0).length+\' background) · reference local frame \'+saved.frame):\'Saved examples: 0 · no reference frame\';\n  [\'low\',\'high\',\'alpha\'].forEach(key=>get(key).value=state[key]);\n  get(\'low\').max=get(\'high\').max=state.maximum;\n  get(\'status\').textContent=state.status;\n  get(\'coverage\').textContent=state.rebuild_required?\'Rebuild project required\':(state.dirty?\'Tracking is incomplete or needs updating\':\'All selected frames tracked\');\n  get(\'progress\').max=state.frame_count;get(\'progress\').value=state.tracked_frames;\n  get(\'legend\').textContent=state.objects.map(item=>item.id+\': \'+item.name).join(\' · \') || \'No objects yet\';\n  const loaded = new Image(); await new Promise((resolve,reject)=>{loaded.onload=resolve;loaded.onerror=()=>reject(new Error(\'Image could not be decoded\'));loaded.src=state.image;});\n  image=loaded;canvas.width=state.width;canvas.height=state.height;zoom();draw();lock(false);\n  if(globalThis.google?.colab?.output?.setIframeHeight) google.colab.output.setIframeHeight(document.documentElement.scrollHeight,true);\n}\nfunction payload() {\n  return {frame:Number(get(\'frame_number\').value), object_id:get(\'object\').value===\'\'?null:Number(get(\'object\').value),\n    concept:get(\'concept\').value, name:get(\'name\').value, name_changed:get(\'name\').value!==state.name,\n    name_prefix:get(\'name_prefix\').value, box_update:get(\'box_update\').value, direction:get(\'direction\').value,\n    low:Number(get(\'low\').value),high:Number(get(\'high\').value),alpha:Number(get(\'alpha\').value),pending:pending};\n}\nasync function request(action) {\n  if(busy || state.closed)return;\n  const data=payload(); lock(true);\n  get(\'status\').textContent=action===\'track\'?\'Tracking selected frames… Wait for completion.\':\'Applying…\';\n  if(action===\'track\')get(\'progress\').removeAttribute(\'value\');\n  try {\n    const response=await google.colab.kernel.invokeFunction(initial.callback,[action,data],{});\n    const result=response.data[\'application/json\'];\n    if(!result || !result.state)throw new Error(\'Invalid response from Python\');\n    if(result.ok && [\'points\',\'concept\',\'exclude\'].includes(action))pending=[];\n    await render(result.state);\n  } catch(error) {get(\'status\').textContent=\'Connection/display error: \'+error.message+\'. Use the latest panel.\';lock(false);}\n}\nget(\'frame\').oninput=()=>{get(\'frame_number\').value=get(\'frame\').value;};\nget(\'frame\').onchange=()=>{pending=[];drag=null;request(\'frame\');};\nget(\'frame_number\').onchange=()=>{pending=[];drag=null;request(\'frame\');};\nget(\'object\').onchange=()=>{pending=[];drag=null;request(\'object\');};\nlet previousTool=get(\'tool\').value;\nget(\'tool\').onchange=()=>{if(previousTool.split(\'_\')[0]!==get(\'tool\').value.split(\'_\')[0])pending=[];previousTool=get(\'tool\').value;drag=null;draw();};\nget(\'zoom\').oninput=()=>{zoom();draw();};\nroot.querySelectorAll(\'[data-action]\').forEach(button=>button.onclick=()=>{\n  if(busy)return;\n  const action=button.dataset.action;\n  if(action===\'undo\'){pending.pop();draw();}\n  else if(action===\'clear\'){pending=[];drag=null;draw();}\n  else if(action===\'track\' && pending.length){get(\'status\').textContent=\'Apply or clear pending annotations before tracking.\';}\n  else request(action);\n});\ncanvas.onpointerdown=event=>{\n  if(busy || state.closed || state.rebuild_required || event.button!==0)return;\n  const position=point(event);if(!inside(position))return;\n  const tool=get(\'tool\').value;\n  if(tool.startsWith(\'point\')){pending.push({kind:\'point\',coordinates:position,label:Number(tool.endsWith(\'positive\'))});draw();}\n  else{drag={start:position,end:position};canvas.setPointerCapture(event.pointerId);}\n};\ncanvas.onpointermove=event=>{if(drag){drag.end=point(event);draw();}};\ncanvas.onpointerup=event=>{\n  if(!drag)return;\n  const end=point(event),start=drag.start;drag=null;\n  if(inside(end)){\n    const left=Math.min(start[0],end[0]),top=Math.min(start[1],end[1]),w=Math.abs(end[0]-start[0]),h=Math.abs(end[1]-start[1]);\n    if(w>0 && h>0)pending.push({kind:\'box\',coordinates:[left/state.width,top/state.height,w/state.width,h/state.height],label:Number(get(\'tool\').value.endsWith(\'positive\'))});\n  }\n  draw();\n};\ncanvas.onpointercancel=()=>{drag=null;draw();};\nroot.dataset.ready=\'loading\';\nreturn render(state).then(()=>{root.dataset.ready=\'true\';return true;});\n\'\'\'\n', 'engine.py': '"""Track selected image frames and export masks with reproducible prompt history."""\n\nfrom __future__ import annotations\n\nimport colorsys\nimport copy\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport re\n\nimport numpy as np\nfrom PIL import Image\nimport tifffile\n\nfrom .io import labels_from_masks\nfrom .labels import LABEL_FORMAT, draw_object_label, mask_folder_name\n\n\nPROJECT_VERSION = 1\nNAMING_POLICY = "concept-prefix-v1"\n\n\ndef _array(value):\n    if hasattr(value, "detach"):\n        value = value.detach().cpu().numpy()\n    return np.asarray(value)\n\n\ndef _integer(value, name, minimum=0):\n    if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < minimum:\n        raise ValueError(f"{name} must be an integer >= {minimum}")\n    return int(value)\n\n\n\n\nclass TrackingProject:\n    """Track and refine detections; use a new project to recover a failed session."""\n\n    def __init__(self, sequence, predictor, prepared, environment=None, max_objects=16):\n        self.sequence = sequence\n        self.predictor = predictor\n        self.prepared = dict(prepared)\n        self.environment = dict(environment or {})\n        self.max_objects = _integer(max_objects, "max_objects", 1)\n        if self.max_objects > 256:\n            raise ValueError("max_objects must be at most 256 and match the predictor configuration")\n        configured_limit = self.environment.get("max_objects")\n        if configured_limit is not None and configured_limit != self.max_objects:\n            raise ValueError("max_objects does not match the loaded predictor")\n        count, height, width = sequence.shape\n        if tuple(prepared.get(key) for key in ("frame_count", "height", "width")) != (count, height, width):\n            raise ValueError("Prepared frames do not match the selected image sequence")\n        for key in ("low", "high"):\n            if not math.isfinite(float(prepared[key])):\n                raise ValueError("Prepared intensity limits must be finite")\n        self.objects = {}\n        self.results = {}\n        self.current_frame_outputs = {}\n        self.current_frame_index = None\n        self.events = []\n        self.dirty = True\n        self.rebuild_required = False\n        self._closed = False\n        self._raw_to_app = {}\n        self._next_app_id = 1\n        self._points = {}\n        self._has_prompts = False\n        self._concept = False\n        self._concept_name = ""\n        self._concept_prompt = None\n        self.name_prefix = ""\n        self._naming_policy = NAMING_POLICY\n        self._coverage = []\n        response = predictor.handle_request({\n            "type": "start_session", "resource_path": prepared["resource_path"],\n            "offload_video_to_cpu": True, "offload_state_to_cpu": False,\n        })\n        self.session_id = response["session_id"]\n\n    def _ready(self):\n        if self._closed:\n            raise RuntimeError("This project is closed")\n        if self.rebuild_required:\n            raise RuntimeError("The predictor session failed; save successful events and replay them in a new project")\n\n    def _frame(self, frame):\n        frame = _integer(frame, "frame")\n        if frame >= self.sequence.shape[0]:\n            raise ValueError("Frame is outside the selected sequence")\n        return frame\n\n    def _object(self, object_id):\n        object_id = _integer(object_id, "object_id", 1)\n        if object_id not in self.objects:\n            raise ValueError("Unknown object ID")\n        return self.objects[object_id]\n\n    def _invalidate(self):\n        self.dirty = True\n        self.results = {}\n        self.current_frame_outputs = {}\n        self.current_frame_index = None\n        self._coverage = []\n\n    def _register(self, raw_id):\n        raw_id = _integer(raw_id, "predictor object ID")\n        if raw_id in self._raw_to_app:\n            return self._raw_to_app[raw_id]\n        if sum(not item["excluded"] for item in self.objects.values()) >= self.max_objects:\n            raise RuntimeError("Predictor object count exceeds the configured limit")\n        app_id = self._next_app_id\n        self._next_app_id += 1\n        color = [round(channel * 255) for channel in colorsys.hsv_to_rgb((app_id * 0.61803398875) % 1, 0.75, 1)]\n        self.objects[app_id] = {"name": f"{self.effective_name_prefix} {app_id}", "color": color,\n                                "raw_id": raw_id, "excluded": False}\n        self._raw_to_app[raw_id] = app_id\n        return app_id\n\n    def _unused_raw_id(self):\n        maximum = max(self._raw_to_app, default=-1)\n        sessions = getattr(self.predictor, "_all_inference_states", {})\n        state = sessions.get(self.session_id, {}).get("state", {})\n        metadata = state.get("tracker_metadata", {})\n        if metadata:\n            maximum = max(maximum, int(metadata.get("max_obj_id", -1)))\n            known_ids = _array(metadata.get("obj_ids_all_gpu", [])).ravel()\n            if known_ids.size:\n                maximum = max(maximum, int(known_ids.max()))\n            if known_ids.size >= self.max_objects:\n                raise ValueError("Configured object limit reached, including currently hidden tracker objects")\n        return maximum + 1\n\n    def _decode(self, response, expected_frame=None):\n        frame = self._frame(response["frame_index"])\n        if expected_frame is not None and frame != expected_frame:\n            raise ValueError("Predictor returned a different prompt frame")\n        outputs = response["outputs"]\n        raw_ids = _array(outputs["out_obj_ids"])\n        masks = _array(outputs["out_binary_masks"])\n        if raw_ids.ndim != 1 or not np.issubdtype(raw_ids.dtype, np.integer):\n            raise ValueError("Predictor IDs must be a one-dimensional integer array")\n        if len(set(raw_ids.tolist())) != len(raw_ids) or np.any(raw_ids < 0):\n            raise ValueError("Predictor IDs must be unique nonnegative integers")\n        if masks.shape != (len(raw_ids), *self.sequence.shape[1:]) or masks.dtype != np.bool_:\n            raise ValueError("Predictor masks must be boolean arrays of shape object,Y,X")\n        for raw_id in sorted(raw_ids.tolist()):\n            self._register(raw_id)\n        result = {}\n        signatures = {}\n        for raw_id, mask in zip(raw_ids.tolist(), masks):\n            app_id = self._raw_to_app[raw_id]\n            signatures[str(raw_id)] = hashlib.sha256(np.packbits(mask).tobytes()).hexdigest()\n            if not self.objects[app_id]["excluded"]:\n                result[app_id] = np.array(mask, dtype=bool, copy=True)\n        return frame, result, signatures\n\n    def _request(self, request, frame, reserve_raw=None):\n        saved_objects = copy.deepcopy(self.objects)\n        saved_mapping = dict(self._raw_to_app)\n        saved_next = self._next_app_id\n        self._invalidate()\n        try:\n            if reserve_raw is not None:\n                self._register(reserve_raw)\n            response = self.predictor.handle_request({**request, "session_id": self.session_id})\n            _, masks, signatures = self._decode(response, frame)\n        except BaseException:\n            self.objects, self._raw_to_app, self._next_app_id = saved_objects, saved_mapping, saved_next\n            self.rebuild_required = True\n            raise\n        self.current_frame_outputs = masks\n        self.current_frame_index = frame\n        return masks, signatures\n\n    def add_concept(self, frame, text, boxes=None, box_labels=None):\n        """Apply the initial text and/or normalized XYWH boxes before point prompts."""\n        self._ready()\n        frame = self._frame(frame)\n        if self._has_prompts:\n            raise ValueError("Use update_concept to redetect after concept or point prompts")\n        text, boxes, box_labels = self._validate_concept(text, boxes, box_labels)\n        return self._apply_concept(frame, text, boxes, box_labels, redetect=False)\n\n    @property\n    def concept_prompt(self):\n        """Return the saved reference frame, text, and normalized example boxes."""\n        return copy.deepcopy(self._concept_prompt)\n\n    def update_concept(self, frame, text, boxes=None, box_labels=None, *, mode="add", name_prefix=None):\n        """Add or replace saved examples, then redetect with fresh object identities."""\n        self._ready()\n        frame = self._frame(frame)\n        if mode not in ("add", "replace"):\n            raise ValueError("Box update must be add or replace")\n        if name_prefix is not None and not isinstance(name_prefix, str):\n            raise ValueError("Name prefix must be a string")\n        boxes = [] if boxes is None else list(boxes)\n        box_labels = [] if box_labels is None else list(box_labels)\n        saved = self._concept_prompt\n        if mode == "add" and saved is not None and saved["boxes"]:\n            if frame != saved["frame"]:\n                raise ValueError(f"Saved examples use reference frame {saved[\'frame\']}; use the reference frame or Replace saved")\n            boxes = saved["boxes"] + boxes\n            box_labels = saved["box_labels"] + box_labels\n        text, boxes, box_labels = self._validate_concept(text, boxes, box_labels)\n        if name_prefix is not None and name_prefix.strip() != self.name_prefix:\n            self.set_name_prefix(name_prefix)\n        return self._apply_concept(frame, text, boxes, box_labels, redetect=self._has_prompts)\n\n    def _validate_concept(self, text, boxes, box_labels):\n        if not isinstance(text, str):\n            raise ValueError("text must be a string")\n        text = text.strip()\n        boxes = [] if boxes is None else boxes\n        box_labels = [] if box_labels is None else box_labels\n        if len(boxes) != len(box_labels):\n            raise ValueError("Each box requires a 0 or 1 label")\n        clean_boxes = []\n        for box in boxes:\n            if len(box) != 4:\n                raise ValueError("Boxes must contain normalized x,y,width,height")\n            x, y, width, height = [float(value) for value in box]\n            if not all(math.isfinite(value) for value in (x, y, width, height)) or not (\n                0 <= x < 1 and 0 <= y < 1 and 0 < width <= 1 - x and 0 < height <= 1 - y\n            ):\n                raise ValueError("Boxes must be finite normalized XYWH coordinates inside the image")\n            clean_boxes.append([x, y, width, height])\n        clean_labels = self._labels(box_labels)\n        if not text and not clean_boxes:\n            raise ValueError("Enter a concept or at least one box")\n        return text, clean_boxes, clean_labels\n\n    def _apply_concept(self, frame, text, boxes, box_labels, *, redetect):\n        request = {"type": "add_prompt", "frame_index": frame}\n        if text:\n            request["text"] = text\n        if boxes:\n            request.update(bounding_boxes=boxes, bounding_box_labels=box_labels)\n        saved_objects = copy.deepcopy(self.objects)\n        saved_mapping, saved_next, saved_points = dict(self._raw_to_app), self._next_app_id, self._points\n        previous_concept_name = self._concept_name\n        if redetect:\n            for item in self.objects.values():\n                item["excluded"] = True\n            self._raw_to_app = {}\n            self._points = {}\n        self._concept_name = text\n        try:\n            masks, signatures = self._request(request, frame)\n        except BaseException:\n            self.objects, self._raw_to_app, self._next_app_id = saved_objects, saved_mapping, saved_next\n            self._points = saved_points\n            self._concept_name = previous_concept_name\n            self._invalidate()\n            self.rebuild_required = True\n            raise\n        self._has_prompts = self._concept = True\n        self._concept_prompt = {"frame": frame, "text": text, "boxes": copy.deepcopy(boxes),\n                                "box_labels": list(box_labels)}\n        self.events.append({"type": "redetect" if redetect else "concept", "frame": frame, "text": text,\n                            "boxes": copy.deepcopy(boxes), "box_labels": list(box_labels), "signatures": signatures})\n        return masks\n\n    @staticmethod\n    def _labels(labels):\n        result = [_integer(value, "prompt label") for value in labels]\n        if any(value not in (0, 1) for value in result):\n            raise ValueError("Prompt labels must be 0 or 1")\n        return result\n\n    def add_points(self, frame, object_id, points, labels, name=""):\n        """Append pixel XY points to this object\'s points on this frame."""\n        self._ready()\n        frame = self._frame(frame)\n        if not isinstance(name, str):\n            raise ValueError("name must be a string")\n        if not len(points) or len(points) != len(labels):\n            raise ValueError("Provide points and one 0 or 1 label per point")\n        clean_labels = self._labels(labels)\n        clean_points = []\n        height, width = self.sequence.shape[1:]\n        for point in points:\n            if len(point) != 2:\n                raise ValueError("Points must contain pixel x,y coordinates")\n            x, y = [float(value) for value in point]\n            if not (math.isfinite(x) and math.isfinite(y) and 0 <= x < width and 0 <= y < height):\n                raise ValueError("Point coordinates must be finite and inside the image")\n            clean_points.append([x, y])\n        is_new = object_id is None\n        if is_new:\n            if sum(not item["excluded"] for item in self.objects.values()) >= self.max_objects:\n                raise ValueError("Configured object limit reached")\n            if 1 not in clean_labels:\n                raise ValueError("A new object requires at least one positive point")\n            raw_id = self._unused_raw_id()\n            object_id = self._next_app_id\n        else:\n            object_id = _integer(object_id, "object_id", 1)\n            item = self._object(object_id)\n            if item["excluded"]:\n                raise ValueError("Excluded objects cannot receive points")\n            raw_id = item["raw_id"]\n        previous_points, previous_labels = self._points.get((frame, object_id), ([], []))\n        combined_points = previous_points + clean_points\n        combined_labels = previous_labels + clean_labels\n        normalized_points = [[x / width, y / height] for x, y in combined_points]\n        request = {"type": "add_prompt", "frame_index": frame, "obj_id": raw_id,\n                   "points": normalized_points, "point_labels": combined_labels,\n                   "clear_old_points": True, "rel_coordinates": True}\n        masks, signatures = self._request(request, frame, raw_id if is_new else None)\n        self._points[frame, object_id] = (combined_points, combined_labels)\n        if name:\n            self.objects[object_id]["name"] = name\n        self._has_prompts = True\n        self.events.append({"type": "points", "frame": frame, "object_id": object_id,\n                            "new_object": is_new, "raw_id": raw_id, "points": clean_points,\n                            "labels": clean_labels, "name": name, "signatures": signatures})\n        return masks\n\n    def remove_object(self, object_id, frame=0):\n        self._ready()\n        frame = self._frame(frame)\n        object_id = _integer(object_id, "object_id", 1)\n        item = self._object(object_id)\n        if item["excluded"]:\n            raise ValueError("Object is already excluded")\n        masks, signatures = self._request({"type": "remove_object", "frame_index": frame,\n                                           "obj_id": item["raw_id"], "is_user_action": False}, frame)\n        self.objects[object_id]["excluded"] = True\n        masks.pop(object_id, None)\n        self.events.append({"type": "remove", "object_id": object_id, "frame": frame,\n                            "signatures": signatures})\n        return masks\n\n    def rename_object(self, object_id, name):\n        """Record a human-assigned object name."""\n        self._ready()\n        object_id = _integer(object_id, "object_id", 1)\n        item = self._object(object_id)\n        if not isinstance(name, str) or not name.strip():\n            raise ValueError("Object name must be nonempty")\n        item.update(name=name.strip())\n        self.events.append({"type": "rename", "object_id": object_id,\n                            "name": name.strip()})\n\n    @property\n    def effective_name_prefix(self):\n        """Use the entered prefix, the concept text, or the default Object label."""\n        concept = self._concept_name if self._naming_policy == NAMING_POLICY else ""\n        return self.name_prefix or concept or "Object"\n\n    def set_name_prefix(self, prefix, *, rename_existing=False):\n        """Name future objects; optionally replace every active object\'s current name."""\n        self._ready()\n        if not isinstance(prefix, str):\n            raise ValueError("Name prefix must be a string")\n        if not isinstance(rename_existing, bool):\n            raise ValueError("rename_existing must be a boolean")\n        prefix = prefix.strip()\n        self.name_prefix = prefix\n        if rename_existing:\n            for object_id, item in self.objects.items():\n                if not item["excluded"]:\n                    item["name"] = f"{self.effective_name_prefix} {object_id}"\n        self.events.append({"type": "set_name_prefix", "prefix": prefix,\n                            "rename_existing": rename_existing})\n\n    def propagate(self, start_frame=0, direction="both", progress=None):\n        """Replace the result snapshot; export requires coverage of every selected frame."""\n        self._ready()\n        start_frame = self._frame(start_frame)\n        if direction not in {"both", "forward", "backward"}:\n            raise ValueError("direction must be both, forward, or backward")\n        if not self._has_prompts:\n            raise ValueError("Add a concept or points before propagation")\n        frame_count = self.sequence.shape[0]\n        expected = set(range(frame_count)) if direction == "both" else (\n            set(range(start_frame, frame_count)) if direction == "forward" else set(range(start_frame + 1)))\n        saved_objects = copy.deepcopy(self.objects)\n        saved_mapping = dict(self._raw_to_app)\n        saved_next = self._next_app_id\n        self._invalidate()\n        pending = {}\n        signatures = {}\n        duplicate_anchor_seen = False\n        try:\n            responses = self.predictor.handle_stream_request({\n                "type": "propagate_in_video", "session_id": self.session_id,\n                "propagation_direction": direction, "start_frame_index": start_frame,\n                "max_frame_num_to_track": frame_count,\n            })\n            for response in responses:\n                frame, masks, signature = self._decode(response)\n                if frame not in expected:\n                    raise ValueError("Predictor returned a frame outside the requested propagation range")\n                if frame in pending:\n                    if direction != "both" or frame != start_frame or duplicate_anchor_seen:\n                        raise ValueError("Predictor returned an unexpected duplicate frame")\n                    duplicate_anchor_seen = True\n                    continue\n                pending[frame] = masks\n                signatures[str(frame)] = signature\n                self.current_frame_outputs = masks\n                self.current_frame_index = frame\n                if progress:\n                    progress(len(pending), len(expected))\n            if set(pending) != expected:\n                raise RuntimeError("Propagation ended before all requested frames were returned")\n        except BaseException:\n            self.objects, self._raw_to_app, self._next_app_id = saved_objects, saved_mapping, saved_next\n            self._invalidate()\n            self.rebuild_required = True\n            raise\n        self.results = dict(sorted(pending.items()))\n        self._coverage = sorted(expected)\n        self.dirty = expected != set(range(frame_count))\n        self.events.append({"type": "propagate", "start_frame": start_frame,\n                            "direction": direction, "signatures": signatures})\n        return self.results\n\n    def _project_data(self):\n        allowed = {"python", "torch", "cuda", "gpu", "free_vram_gib", "total_vram_gib",\n                   "sam_revision", "checkpoint_revision", "checkpoint_file", "max_objects",\n                   "flash_attention_3", "compile", "packages"}\n        environment = {key: value for key, value in self.environment.items() if key in allowed}\n        if "packages" in environment:\n            packages = environment["packages"]\n            environment["packages"] = {\n                key: value for key, value in packages.items()\n                if isinstance(key, str) and re.fullmatch(r"[A-Za-z0-9_.-]+", key)\n                and isinstance(value, str) and re.fullmatch(r"[A-Za-z0-9.+!_-]+", value)\n            } if isinstance(packages, dict) else {}\n        for key, value in list(environment.items()):\n            if isinstance(value, str) and ("/" in value or "\\\\" in value):\n                environment.pop(key)\n        return {"schema": "sam31-colab-project", "version": PROJECT_VERSION,\n                "naming_policy": self._naming_policy,\n                "fingerprint": self.sequence.fingerprint(),\n                "prepared": {key: self.prepared[key] for key in ("low", "high", "frame_count", "height", "width")},\n                "max_objects": self.max_objects, "environment": environment,\n                "events": copy.deepcopy(self.events),\n                "objects": {str(key): copy.deepcopy(value) for key, value in self.objects.items()},\n                "coverage": self._coverage, "complete": not self.dirty,\n                "overlap_policy": "lowest positive object ID wins in overlays; individual masks are preserved",\n                "duplicate_anchor_policy": "first (forward) output wins",\n                "point_policy": "append per object and frame; resend accumulated points with clear_old_points=true"}\n\n    def save_project(self, path, *, export_info=None):\n        """Save JSON without input/cache absolute paths or authentication values."""\n        destination = Path(path).expanduser()\n        self._safe_output(destination)\n        data = self._project_data()\n        if export_info is not None:\n            data["export"] = copy.deepcopy(export_info)\n        serialized = json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False)\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        with destination.open("x", encoding="utf-8") as stream:\n            stream.write(serialized)\n        return destination\n\n    def replay_project(self, path):\n        """Replay on a fresh project; exact mask identity checks can reject nondeterministic inference."""\n        self._ready()\n        if self.events or self.objects:\n            raise ValueError("Replay requires a new empty project")\n        with Path(path).open("r", encoding="utf-8") as stream:\n            data = json.load(stream)\n        if data.get("schema") != "sam31-colab-project" or data.get("version") != PROJECT_VERSION:\n            raise ValueError("Unsupported project schema or version")\n        naming_policy = data.get("naming_policy", "legacy")\n        if not isinstance(naming_policy, str) or naming_policy not in (NAMING_POLICY, "legacy"):\n            raise ValueError("Unsupported project naming policy")\n        saved_environment = data.get("environment", {})\n        for key in ("sam_revision", "checkpoint_revision"):\n            if key in saved_environment and key in self.environment and saved_environment[key] != self.environment[key]:\n                raise ValueError(f"Model environment differs from the saved project: {key}")\n        if data.get("fingerprint") != self.sequence.fingerprint():\n            raise ValueError("Input fingerprint or image settings differ from the saved project")\n        expected_prepared = {key: self.prepared[key] for key in ("low", "high", "frame_count", "height", "width")}\n        if data.get("prepared") != expected_prepared or data.get("max_objects") != self.max_objects:\n            raise ValueError("Prepared intensity settings or object limit differ from the saved project")\n        self._naming_policy = naming_policy\n        try:\n            for event in data["events"]:\n                kind = event["type"]\n                if kind == "concept":\n                    self.add_concept(event["frame"], event["text"], event["boxes"], event["box_labels"])\n                elif kind == "redetect":\n                    self.update_concept(event["frame"], event["text"], event["boxes"], event["box_labels"], mode="replace")\n                elif kind == "points":\n                    if event["new_object"]:\n                        if event["object_id"] != self._next_app_id or event["raw_id"] != self._unused_raw_id():\n                            raise ValueError("New-object identity differs from the saved project")\n                    elif self._object(event["object_id"])["raw_id"] != event["raw_id"]:\n                        raise ValueError("Point target identity differs from the saved project")\n                    self.add_points(event["frame"], None if event["new_object"] else event["object_id"],\n                                    event["points"], event["labels"], event["name"])\n                elif kind == "remove":\n                    self.remove_object(event["object_id"], event["frame"])\n                elif kind == "rename":\n                    self.rename_object(event["object_id"], event["name"])\n                elif kind == "set_name_prefix":\n                    self.set_name_prefix(event["prefix"], rename_existing=event["rename_existing"])\n                elif kind == "propagate":\n                    self.propagate(event["start_frame"], event["direction"])\n                else:\n                    raise ValueError("Unknown project event")\n                if self.events[-1] != event:\n                    raise ValueError("Replay object IDs or masks differ; refusing to apply subsequent prompts to uncertain identities")\n            if {str(key): value for key, value in self.objects.items()} != data["objects"]:\n                raise ValueError("Replayed object metadata differs from the saved project")\n        except BaseException:\n            self._invalidate()\n            self.rebuild_required = True\n            raise\n        return self\n\n    load_project = replay_project\n\n    def _safe_output(self, destination):\n        target = destination.resolve()\n        source = Path(self.sequence.settings.input_path).expanduser().resolve()\n        cache = Path(self.prepared["resource_path"]).expanduser().resolve()\n        protected = [(source, source.is_dir()), (cache, True)]\n        protected.extend((Path(path).resolve(), False) for path in getattr(self.sequence, "_files", []))\n        for item, directory in protected:\n            if target == item or target in item.parents or (directory and item in target.parents):\n                raise ValueError("Output must not overlap the original input or prepared-frame cache")\n\n    def export(self, output_dir, output_mode="mask"):\n        """Export segmentation images and one JSON containing prompts and reproducibility settings."""\n        self._ready()\n        if self.dirty or set(self.results) != set(range(self.sequence.shape[0])):\n            raise RuntimeError("Run complete propagation after all corrections before exporting")\n        if output_mode not in ("mask", "overlay", "both"):\n            raise ValueError(\'output_mode must be "mask", "overlay", or "both"\')\n        save_masks = output_mode in ("mask", "both")\n        save_overlays = output_mode in ("overlay", "both")\n        destination = Path(output_dir).expanduser()\n        self._safe_output(destination)\n        if destination.exists():\n            if not destination.is_dir() or any(destination.iterdir()):\n                raise FileExistsError("Choose an empty or new export directory; existing files will not be overwritten")\n        else:\n            destination.mkdir(parents=True, exist_ok=False)\n        if save_masks:\n            (destination / "masks").mkdir()\n        if save_overlays:\n            (destination / "overlays").mkdir()\n        active_ids = sorted(key for key, item in self.objects.items() if not item["excluded"])\n        mask_folders = {object_id: Path("masks") / mask_folder_name(object_id, self.objects[object_id])\n                        for object_id in active_ids} if save_masks else {}\n        if save_masks:\n            for object_id in active_ids:\n                (destination / mask_folders[object_id]).mkdir()\n        zero_mask = np.zeros(self.sequence.shape[1:], dtype=bool)\n        for local_frame, masks in sorted(self.results.items()):\n            source_frame = self.sequence.frame_indices[local_frame]\n            full_masks = {object_id: masks.get(object_id, zero_mask) for object_id in active_ids}\n            if save_overlays:\n                frame_labels = (labels_from_masks(active_ids, list(full_masks.values())) if active_ids\n                                else np.zeros(self.sequence.shape[1:], dtype=np.uint16))\n                overlay = self.sequence.preview(source_frame, low=self.prepared["low"], high=self.prepared["high"])\n            for object_id, mask in full_masks.items():\n                if save_masks:\n                    tifffile.imwrite(destination / mask_folders[object_id] / f"frame_{local_frame:06d}.tif",\n                                     mask.astype(np.uint8) * 255, metadata={"axes": "YX"})\n                if save_overlays:\n                    visible = frame_labels == object_id\n                    overlay[visible] = np.rint(0.55 * overlay[visible] + 0.45 * np.array(self.objects[object_id]["color"])).astype(np.uint8)\n            if save_overlays:\n                overlay_image = Image.fromarray(overlay)\n                for object_id, mask in full_masks.items():\n                    draw_object_label(overlay_image, mask, object_id, self.objects[object_id])\n                overlay_image.save(destination / "overlays" / f"frame_{local_frame:06d}.png")\n        self.save_project(destination / "project.json", export_info={\n            "complete": True, "output_mode": output_mode,\n            "source_frame_indices": list(self.sequence.frame_indices),\n            "frame_filename": "frame_{local_frame_index:06d}",\n            "mask_values": {"background": 0, "foreground": 255},\n            "overlay_opacity": 0.45, "overlay_object_ids": True,\n            "mask_folders": {str(key): value.as_posix() for key, value in mask_folders.items()},\n            "overlay_label_format": LABEL_FORMAT,\n        })\n        return destination\n\n    def close(self):\n        if not self._closed:\n            self.predictor.handle_request({"type": "close_session", "session_id": self.session_id})\n            self._closed = True\n', 'io.py': '"""Read TIFF and grayscale raster images and prepare model inputs and previews."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nimport hashlib\nimport json\nimport math\nimport re\nfrom pathlib import Path\nfrom typing import Sequence\n\nimport numpy as np\nfrom PIL import Image\nimport tifffile\n\n\n@dataclass\nclass ImageSettings:\n    input_path: str\n    axes: str = ""\n    series: int = 0\n    start: int = 0\n    stop: int | None = None\n    step: int = 1\n    intensity_low: float | None = None\n    intensity_high: float | None = None\n\n\ndef _natural_key(path: Path) -> list[object]:\n    return [int(part) if part.isdigit() else part.casefold() for part in re.split(r"(\\d+)", path.name)]\n\n\nclass TiffSequence:\n    """Read TIFF sequences or raster frames; shape is (time, height, width)."""\n\n    _SUPPORTED_AXES = set("TCZYXS")\n    _TIFF_EXTENSIONS = {".tif", ".tiff"}\n    _RASTER_EXTENSIONS = {".jpg", ".jpeg", ".png"}\n\n    def __init__(self, settings: ImageSettings):\n        self.settings = settings\n        source = Path(settings.input_path).expanduser()\n        if not source.exists():\n            raise FileNotFoundError(f"Input path does not exist: {source}")\n        if settings.step <= 0:\n            raise ValueError("step must be greater than zero")\n        if settings.start < 0 or (settings.stop is not None and settings.stop < 0):\n            raise ValueError("start and stop must be non-negative")\n        if source.is_dir():\n            files = sorted(\n                (path for path in source.iterdir()\n                 if path.is_file() and path.suffix.casefold() in self._TIFF_EXTENSIONS | self._RASTER_EXTENSIONS),\n                key=lambda path: (_natural_key(path), path.name),\n            )\n            if not files:\n                raise ValueError("The input directory contains no supported TIFF, JPG/JPEG, or PNG files")\n        elif source.is_file() and source.suffix.casefold() in self._TIFF_EXTENSIONS | self._RASTER_EXTENSIONS:\n            files = [source]\n        else:\n            raise ValueError("Input must be a TIFF, JPG/JPEG, or PNG file or a directory of supported images")\n\n        self._files = files\n        self._file_info: list[dict[str, object]] = []\n        self._arrays: dict[int, np.ndarray] = {}\n        self._offsets: list[int] = []\n        common_height: int | None = None\n        common_width: int | None = None\n        common_dtype: str | None = None\n        common_color_mode: str | None = None\n        frame_total = 0\n\n        for path in files:\n            info = self._inspect_file(path) if path.suffix.casefold() in self._TIFF_EXTENSIONS else self._inspect_raster_file(path)\n            self._offsets.append(frame_total)\n            frame_total += int(info["frame_count"])\n            if common_height is None:\n                common_height, common_width = int(info["height"]), int(info["width"])\n                common_dtype = str(info["dtype"])\n                common_color_mode = str(info["color_mode"])\n            elif (int(info["height"]) != common_height or int(info["width"]) != common_width\n                  or str(info["dtype"]) != common_dtype or str(info["color_mode"]) != common_color_mode):\n                raise ValueError("All image files in a sequence must have matching XY dimensions, dtype, and color mode")\n            self._file_info.append(info)\n\n        selected_stop = frame_total if settings.stop is None else min(settings.stop, frame_total)\n        if settings.start >= frame_total or selected_stop <= settings.start:\n            raise ValueError("The requested frame range contains no frames")\n        self.frame_indices = tuple(range(settings.start, selected_stop, settings.step))\n        self.shape = (len(self.frame_indices), int(common_height), int(common_width))\n        self._source_frame_count = frame_total\n        self._dtype = str(common_dtype)\n        self.is_rgb = common_color_mode == "RGB"\n        axes_by_file = [str(info["axes"]) for info in self._file_info]\n        self.metadata = {\n            "axes": axes_by_file[0] if len(set(axes_by_file)) == 1 else axes_by_file,\n            "source_names": [path.name for path in files],\n            "dtype": self._dtype,\n            "color_mode": common_color_mode,\n            "source_frame_count": frame_total,\n            "selected_frame_indices": list(self.frame_indices),\n        }\n\n    def _inspect_file(self, path: Path) -> dict[str, object]:\n        with tifffile.TiffFile(path) as tif:\n            if not 0 <= self.settings.series < len(tif.series):\n                raise ValueError(f"series index {self.settings.series} is outside the available range for {path.name}")\n            series = tif.series[self.settings.series]\n            shape = tuple(int(v) for v in series.shape)\n            stored_dimensions = dict(zip(str(series.axes).upper(), shape))\n            axes = self.settings.axes.upper() if self.settings.axes else str(series.axes).upper()\n            if len(axes) != len(shape):\n                raise ValueError(f"Axes \'{axes}\' do not match the {len(shape)} dimensions in {path.name}")\n            if len(set(axes)) != len(axes):\n                raise ValueError(f"Axes \'{axes}\' contain duplicate labels")\n            dimensions = dict(zip(axes, shape))\n            if stored_dimensions.get("Z", 1) > 1 or dimensions.get("Z", 1) > 1:\n                raise ValueError("Z stacks are not supported. Use a 2D image or a 2D time-lapse sequence; do not relabel depth as time.")\n            if stored_dimensions.get("C", 1) > 1 or dimensions.get("C", 1) > 1:\n                raise ValueError("Only single-channel grayscale or RGB TIFF input is supported. Non-RGB multichannel images are not supported; do not relabel channels as time.")\n            first_page = series.pages[0]\n            is_rgb = first_page.photometric == tifffile.PHOTOMETRIC.RGB\n            if is_rgb:\n                if first_page.samplesperpixel != 3 or first_page.extrasamples or stored_dimensions.get("S") != 3:\n                    raise ValueError("RGB TIFF input must contain exactly three color samples and no alpha channel.")\n                if "S" not in axes or axes.index("S") != str(series.axes).upper().index("S"):\n                    raise ValueError("Keep the RGB sample axis S in its stored position; only override unknown time axes.")\n            elif (first_page.samplesperpixel != 1 or dimensions.get("S", 1) > 1 or stored_dimensions.get("S", 1) > 1\n                  or first_page.photometric not in (tifffile.PHOTOMETRIC.MINISBLACK, tifffile.PHOTOMETRIC.MINISWHITE)):\n                raise ValueError("Use single-channel grayscale TIFFs or TIFFs tagged as RGB; color samples cannot be inferred from an axis override.")\n            unsupported = set(axes) - self._SUPPORTED_AXES\n            if unsupported:\n                if not self.settings.axes:\n                    raise ValueError(f"Ambiguous or unsupported TIFF axes {axes!r} in {path.name}; provide an explicit axes override")\n                raise ValueError(f"Unsupported axis labels {sorted(unsupported)} in explicit axes override")\n            if "Y" not in axes or "X" not in axes:\n                raise ValueError(f"TIFF axes must include Y and X in {path.name}")\n            dtype = np.dtype(series.dtype)\n            if dtype not in (np.dtype("uint8"), np.dtype("uint16")):\n                raise ValueError(f"Only uint8 and uint16 TIFF input is supported; {path.name} has {dtype}")\n            return {\n                "shape": shape,\n                "axes": axes,\n                "frame_count": dimensions.get("T", 1),\n                "height": dimensions["Y"],\n                "width": dimensions["X"],\n                "dtype": dtype.name,\n                "color_mode": "RGB" if is_rgb else "grayscale",\n            }\n\n    def _inspect_raster_file(self, path: Path) -> dict[str, object]:\n        if self.settings.series != 0:\n            raise ValueError(f"Raster image series must be 0 for {path.name}")\n        axes = self.settings.axes.upper()\n        if axes not in ("", "YX"):\n            raise ValueError(f"Raster image axes must be empty or \'YX\' for {path.name}")\n\n        extension = path.suffix.casefold()\n        try:\n            with Image.open(path) as image:\n                expected_format = {".jpg": "JPEG", ".jpeg": "JPEG", ".png": "PNG"}[extension]\n                if image.format != expected_format:\n                    raise ValueError(\n                        f"Image format {image.format!r} does not match the {extension} extension for {path.name}"\n                    )\n                if expected_format == "PNG" and (getattr(image, "is_animated", False) or getattr(image, "n_frames", 1) != 1):\n                    raise ValueError(f"Animated PNG images are not supported: {path.name}")\n                if expected_format == "PNG" and ("transparency" in image.info or "tRNS" in image.info):\n                    raise ValueError(f"PNG transparency is not supported: {path.name}")\n                grayscale_modes = {"1", "L", "I", "I;16", "I;16B", "I;16L"}\n                if image.mode not in grayscale_modes | {"RGB"}:\n                    raise ValueError(\n                        f"Only grayscale or RGB JPG/JPEG and PNG images are supported; {path.name} has mode {image.mode!r}"\n                    )\n                if expected_format == "JPEG" and image.mode not in {"L", "RGB"}:\n                    raise ValueError(\n                        f"JPEG input must be 8-bit grayscale or RGB; {path.name} has mode {image.mode!r}"\n                    )\n                if expected_format == "PNG" and image.mode in {"I", "I;16", "I;16B", "I;16L"}:\n                    pixels = np.asarray(image)\n                    if pixels.size and (int(pixels.min()) < 0 or int(pixels.max()) > 65535):\n                        raise ValueError(f"16-bit grayscale PNG values must be within 0..65535: {path.name}")\n                    dtype = "uint16"\n                else:\n                    dtype = "uint8"\n                height, width = image.height, image.width\n                color_mode = "RGB" if image.mode == "RGB" else "grayscale"\n        except (OSError, SyntaxError) as error:\n            raise ValueError(f"Could not read raster image {path.name}: {error}") from error\n        return {\n            "shape": (height, width),\n            "axes": "YXS" if color_mode == "RGB" else "YX",\n            "frame_count": 1,\n            "height": height,\n            "width": width,\n            "dtype": dtype,\n            "color_mode": color_mode,\n            "format": "raster",\n            "mode": image.mode,\n        }\n\n\n    def _load_file(self, file_index: int) -> np.ndarray:\n        if file_index not in self._arrays:\n            self._arrays.clear()\n            path = self._files[file_index]\n            if self._file_info[file_index].get("format") == "raster":\n                with Image.open(path) as image:\n                    arr = np.asarray(image)\n                if arr.dtype.kind == "i":\n                    arr = arr.astype(np.uint16)\n                if self._file_info[file_index].get("mode") == "1":\n                    arr = arr.astype(np.uint8) * 255\n                canonical = arr[None, None, None, :, :, :] if self._file_info[file_index]["color_mode"] == "RGB" else arr[None, None, None, :, :, None]\n            else:\n                with tifffile.TiffFile(path) as tif:\n                    arr = tif.series[self.settings.series].asarray()\n                axes = str(self._file_info[file_index]["axes"])\n                present = list(axes)\n                for axis in "TCZYXS":\n                    if axis not in present:\n                        arr = np.expand_dims(arr, axis=arr.ndim)\n                        present.append(axis)\n                permutation = [present.index(axis) for axis in "TCZYXS"]\n                canonical = np.transpose(arr, permutation)\n            self._arrays[file_index] = canonical\n        return self._arrays[file_index]\n\n    def _locate(self, original_index: int) -> tuple[int, int]:\n        if original_index < 0 or original_index >= self._source_frame_count:\n            raise IndexError(f"frame index {original_index} is outside the source range")\n        file_index = len(self._offsets) - 1\n        for candidate, offset in enumerate(self._offsets):\n            end = offset + int(self._file_info[candidate]["frame_count"])\n            if offset <= original_index < end:\n                file_index = candidate\n                break\n        return file_index, original_index - self._offsets[file_index]\n\n    def raw_frame(self, index: int) -> np.ndarray:\n        """Return raw YX grayscale or YXS RGB values in the original integer dtype."""\n        file_index, local_time = self._locate(int(index))\n        info = self._file_info[file_index]\n        if info.get("format") == "raster":\n            frame = self._load_file(file_index)[0, 0, 0]\n            return frame if self.is_rgb else frame[..., 0]\n        axes = str(info["axes"])\n        shape = tuple(info["shape"])\n        pixel_axes = ("YXS" if axes.endswith("YXS") else "SYX") if self.is_rgb else "YX"\n        if axes.endswith(pixel_axes):\n            with tifffile.TiffFile(self._files[file_index]) as tif:\n                pages = tif.series[self.settings.series].pages\n                leading_shape = shape[:-len(pixel_axes)]\n                page_count = math.prod(leading_shape)\n                if len(pages) == page_count and tuple(pages[0].shape) == shape[-len(pixel_axes):]:\n                    selected = {"T": local_time, "C": 0, "Z": 0, "S": 0}\n                    indices = tuple(selected[axis] for axis in axes[:-len(pixel_axes)])\n                    page_index = int(np.ravel_multi_index(indices, leading_shape)) if leading_shape else 0\n                    frame = pages[page_index].asarray()\n                    return np.moveaxis(frame, 0, -1) if pixel_axes == "SYX" else frame\n        frame = np.asarray(self._load_file(file_index)[local_time, 0, 0])\n        return frame if self.is_rgb else frame[..., 0]\n\n    @staticmethod\n    def _scale_to_rgb(frame: np.ndarray, low: float, high: float) -> np.ndarray:\n        if high <= low:\n            pixels = np.zeros(frame.shape, dtype=np.uint8)\n        else:\n            scaled = ((np.asarray(frame, dtype=np.float64) - low) / (high - low)) * 255.0\n            pixels = np.clip(np.rint(scaled), 0, 255).astype(np.uint8)\n        return pixels if frame.ndim == 3 else np.repeat(pixels[:, :, None], 3, axis=2)\n\n    def preview(self, index: int, low: float | None = None, high: float | None = None) -> np.ndarray:\n        if low is None:\n            low = self.settings.intensity_low\n        if high is None:\n            high = self.settings.intensity_high\n        if low is None or high is None:\n            sample_low, sample_high = self._sample_limits()\n            low = sample_low if low is None else low\n            high = sample_high if high is None else high\n        if not math.isfinite(low) or not math.isfinite(high):\n            raise ValueError("Display intensity limits must be finite")\n        if high < low:\n            raise ValueError("high must be greater than or equal to low")\n        return self._scale_to_rgb(self.raw_frame(index), float(low), float(high))\n\n    def _sample_limits(self) -> tuple[float, float]:\n        if self.settings.intensity_low is not None and self.settings.intensity_high is not None:\n            return float(self.settings.intensity_low), float(self.settings.intensity_high)\n        if self.is_rgb:\n            return 0.0, float(np.iinfo(self._dtype).max)\n        sampled: list[np.ndarray] = []\n        count = min(32, len(self.frame_indices))\n        positions = np.linspace(0, len(self.frame_indices) - 1, count, dtype=int)\n        for position in np.unique(positions):\n            frame = self.raw_frame(self.frame_indices[int(position)])\n            flat = frame.ravel()\n            stride = max(1, math.ceil(flat.size / 65536))\n            sampled.append(flat[::stride].astype(np.float64, copy=False))\n        if not sampled:\n            return 0.0, 0.0\n        values = np.concatenate(sampled)\n        computed_low, computed_high = (float(v) for v in np.percentile(values, [1.0, 99.8]))\n        return computed_low, computed_high\n\n    def prepare(self, cache_dir: str | Path) -> dict[str, object]:\n        destination = Path(cache_dir).expanduser()\n        if destination.exists():\n            raise FileExistsError(f"Cache path already exists: {destination}")\n        low = self.settings.intensity_low\n        high = self.settings.intensity_high\n        if low is None or high is None:\n            sampled_low, sampled_high = self._sample_limits()\n            low = sampled_low if low is None else float(low)\n            high = sampled_high if high is None else float(high)\n        low, high = float(low), float(high)\n        if high < low:\n            raise ValueError("intensity_high must be greater than or equal to intensity_low")\n        if not math.isfinite(low) or not math.isfinite(high):\n            raise ValueError("Intensity limits must be finite")\n        destination.mkdir(parents=True, exist_ok=False)\n        for output_index, source_index in enumerate(self.frame_indices):\n            rgb = self._scale_to_rgb(self.raw_frame(source_index), low, high)\n            Image.fromarray(rgb, mode="RGB").save(destination / f"{output_index:06d}.png", format="PNG")\n        return {\n            "resource_path": str(destination.resolve()),\n            "low": low,\n            "high": high,\n            "frame_count": len(self.frame_indices),\n            "height": self.shape[1],\n            "width": self.shape[2],\n        }\n\n    def fingerprint(self) -> dict[str, object]:\n        sources = []\n        for path in self._files:\n            digest = hashlib.sha256()\n            with path.open("rb") as stream:\n                for block in iter(lambda: stream.read(1024 * 1024), b""):\n                    digest.update(block)\n            sources.append({"name": path.name, "size_bytes": path.stat().st_size, "sha256": digest.hexdigest()})\n        settings = asdict(self.settings)\n        settings["input_path"] = Path(self.settings.input_path).name\n        payload = {"settings": settings, "sources": sources}\n        stable_json = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)\n        return {**payload, "sha256": hashlib.sha256(stable_json.encode("utf-8")).hexdigest()}\n\n\ndef labels_from_masks(object_ids: Sequence[int], masks: Sequence[np.ndarray]) -> np.ndarray:\n    """Combine same-shaped boolean masks into an integer label image."""\n    if len(object_ids) != len(masks):\n        raise ValueError("object_ids and masks must have the same length")\n    if any(isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) for value in object_ids):\n        raise ValueError("object IDs must be integers")\n    ids = [int(value) for value in object_ids]\n    if any(value <= 0 for value in ids):\n        raise ValueError("object IDs must be positive integers")\n    if len(set(ids)) != len(ids):\n        raise ValueError("object IDs must be unique")\n    if any(value > np.iinfo(np.uint32).max for value in ids):\n        raise ValueError("object IDs must not exceed the uint32 maximum")\n    shape: tuple[int, ...] | None = None\n    normalized: list[tuple[int, np.ndarray]] = []\n    for object_id, mask in zip(ids, masks):\n        array = np.asarray(mask)\n        if array.ndim != 2:\n            raise ValueError("Each mask must be a two-dimensional YX array")\n        if array.dtype != np.bool_:\n            raise ValueError("Each mask must have boolean dtype")\n        if shape is None:\n            shape = array.shape\n        elif array.shape != shape:\n            raise ValueError("All masks must have matching shapes")\n        normalized.append((object_id, array))\n    if shape is None:\n        raise ValueError("At least one mask is required to determine the output shape")\n    maximum_id = max(ids)\n    dtype = np.uint16 if maximum_id <= np.iinfo(np.uint16).max else np.uint32\n    labels = np.zeros(shape, dtype=dtype)\n    for object_id, mask in sorted(normalized, key=lambda pair: pair[0]):\n        labels[mask & (labels == 0)] = object_id\n    return labels\n', 'labels.py': '"""Render current object names and build portable, ID-qualified output paths."""\n\nfrom functools import lru_cache\nfrom pathlib import Path\nimport re\nimport warnings\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\n\n\nLABEL_FORMAT = "{object_id}: {name} (default Object {object_id} names display the ID only)"\n\n\ndef object_label(object_id, item):\n    """Read the current name without changing the stable application ID."""\n    name = item["name"]\n    return str(object_id) if name == f"Object {object_id}" else f"{object_id}: {name}"\n\n\ndef mask_folder_name(object_id, item):\n    """Return one portable path component, bounded in characters and UTF-8 bytes."""\n    prefix = f"object_{object_id:06d}"\n    if item["name"] == f"Object {object_id}":\n        return prefix\n    name = re.sub(r\'[<>:"/\\\\|?*\\x00-\\x1f\\x7f-\\x9f]\', "_", item["name"])\n    name = re.sub(r"\\s+", "_", name.strip(" ."))[:80]\n    name = name.encode("utf-8")[:160].decode("utf-8", errors="ignore").rstrip(" .")\n    return f"{prefix}_{name or \'unnamed\'}"\n\n\n@lru_cache(maxsize=1)\ndef _font_paths():\n    # Use an installed system font supporting the object names.\n    from matplotlib import font_manager\n\n    paths = set(font_manager.findSystemFonts())\n    paths.add(font_manager.findfont("DejaVu Sans"))\n    preferred = ("meiryo", "yugoth", "msgothic", "noto", "ipa", "arialuni", "dejavu")\n\n    def priority(path):\n        name = Path(path).name.lower()\n        return next((index for index, token in enumerate(preferred) if token in name), len(preferred)), name\n\n    return tuple(sorted(paths, key=priority))\n\n\n@lru_cache(maxsize=128)\ndef _font_path(text):\n    from matplotlib.ft2font import FT2Font\n\n    required = {ord(character) for character in text if not character.isspace()}\n    best_path, best_coverage = None, -1\n    for path in _font_paths():\n        try:\n            characters = FT2Font(path).get_charmap()\n        except (OSError, RuntimeError, ValueError):\n            continue\n        coverage = len(required.intersection(characters))\n        if coverage == len(required):\n            return path\n        if coverage > best_coverage:\n            best_path, best_coverage = path, coverage\n    warnings.warn("Available fonts do not support every object-name character; "\n                  "unsupported characters may appear as missing glyphs. Install a suitable system font.",\n                  RuntimeWarning, stacklevel=2)\n    return best_path\n\n\ndef draw_object_label(image, mask, object_id, item, *, selected=False):\n    """Draw a white outlined name, scaling and placing the full label inside the canvas."""\n    rows, columns = np.nonzero(mask)\n    if not len(rows):\n        return\n    text = object_label(object_id, item)\n    # Keep each name on one line, including names supplied through Apply points.\n    text = " ".join(text.splitlines())\n    font_size = max(12, min(24, round(min(image.size) / 32))) + (2 if selected else 0)\n    path = _font_path(text)\n    font = ImageFont.truetype(path, font_size) if path else ImageFont.load_default()\n    drawing = ImageDraw.Draw(image)\n    try:\n        left, top, right, bottom = drawing.textbbox((0, 0), text, font=font, stroke_width=1)\n    except UnicodeEncodeError:\n        warnings.warn("The fallback font cannot render this object name; the image label uses an explicit placeholder.",\n                      RuntimeWarning, stacklevel=2)\n        text = f"{object_id}: [unsupported name]"\n        left, top, right, bottom = drawing.textbbox((0, 0), text, font=font, stroke_width=1)\n    label = Image.new("RGBA", (max(1, right - left), max(1, bottom - top)))\n    ImageDraw.Draw(label).text((-left, -top), text, font=font, fill="white",\n                               stroke_width=1, stroke_fill="black")\n    width, height = image.size\n    margin = 2 if min(width, height) > 4 else 0\n    scale = min(1, (width - 2 * margin) / label.width, (height - 2 * margin) / label.height)\n    if scale < 1:\n        label = label.resize((max(1, int(label.width * scale)), max(1, int(label.height * scale))),\n                             Image.Resampling.LANCZOS)\n    x = min(max(margin, round(float(columns.mean()) - label.width / 2)), width - margin - label.width)\n    y = min(max(margin, round(float(rows.mean()) - label.height / 2)), height - margin - label.height)\n    image.paste(label, (x, y), label)\n', 'runtime.py': '"""Check your GPU and load the SAM 3.1 checkpoint."""\n\nfrom functools import wraps\nfrom importlib import metadata\nimport inspect\nimport json\nfrom pathlib import Path\nimport sys\n\nSAM_REVISION = "7f473cfd983d5eb43040ff608f66bff0d7bdcb53"\nMODEL_REPOSITORY = "facebook/sam3.1"\nMODEL_FILENAME = "sam3.1_multiplex.pt"\n\n\ndef check_python_version():\n    """Require the Python version supported by this notebook\'s pinned NumPy."""\n    if sys.version_info[:2] != (3, 12):\n        raise RuntimeError(\n            "This notebook requires Python 3.12. In Colab, choose Runtime > Change runtime type > "\n            "Runtime Version and select Python 3.12, then rerun section 1 and restart the session."\n        )\n\n\ndef verify_sam_revision():\n    """Run the installation cell if the installed SAM source does not match this notebook."""\n    try:\n        direct_url = json.loads(metadata.distribution("sam3").read_text("direct_url.json") or "{}")\n        revision = direct_url.get("vcs_info", {}).get("commit_id")\n    except (metadata.PackageNotFoundError, ValueError):\n        revision = None\n    if revision != SAM_REVISION:\n        raise RuntimeError("The installed SAM source revision differs. Run section 1, restart the session, and continue at section 2.")\n    return revision\n\n\ndef check_environment():\n    """Use a CUDA GPU with native BF16 support and a compatible PyTorch build."""\n    check_python_version()\n    import torch\n    from packaging.version import Version\n\n    if Version(torch.__version__.split("+")[0]) < Version("2.7"):\n        raise RuntimeError("Use PyTorch 2.7 or later with a matching torchvision build.")\n    if not torch.cuda.is_available():\n        raise RuntimeError("Select Runtime > Change runtime type > GPU, then reconnect.")\n    if Version(torch.version.cuda or "0") < Version("12.6"):\n        raise RuntimeError("Use a PyTorch CUDA build of version 12.6 or later.")\n    if torch.cuda.get_device_capability()[0] < 8 or not torch.cuda.is_bf16_supported():\n        raise RuntimeError("This notebook requires native BF16 support. Select an A100 or another compatible GPU; T4 is not supported.")\n    verify_sam_revision()\n    free_bytes, total_bytes = torch.cuda.mem_get_info()\n    return {\n        "python": sys.version.split()[0], "torch": torch.__version__,\n        "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(),\n        "free_vram_gib": round(free_bytes / 2**30, 2),\n        "total_vram_gib": round(total_bytes / 2**30, 2),\n        "sam_revision": SAM_REVISION,\n    }\n\n\ndef adapt_session_arguments(predictor):\n    """Keep frame offloading enabled; state offloading is unavailable for this model."""\n    original = predictor.model.init_state\n    parameters = inspect.signature(original).parameters\n    if "offload_state_to_cpu" in parameters or any(\n        item.kind == inspect.Parameter.VAR_KEYWORD for item in parameters.values()\n    ):\n        return predictor\n\n    @wraps(original)\n    def initialize(*args, offload_state_to_cpu=False, **kwargs):\n        if offload_state_to_cpu:\n            raise ValueError("SAM 3.1 does not support CPU offloading of tracker state in this notebook.")\n        return original(*args, **kwargs)\n\n    predictor.model.init_state = initialize\n    return predictor\n\n\ndef adapt_point_output(predictor):\n    """Let SAM merge refined masks when the first point prompt has no cached frame."""\n    model = predictor.model\n    original = model._build_sam2_output\n    if getattr(original, "_first_point_output_fixed", False):\n        return predictor\n\n    @wraps(original)\n    def build_output(inference_state, frame_idx, refined_obj_id_to_mask=None):\n        cached_outputs = inference_state["cached_frame_outputs"]\n        if frame_idx in cached_outputs or not refined_obj_id_to_mask:\n            return original(inference_state, frame_idx, refined_obj_id_to_mask)\n\n        patched_state = inference_state.copy()\n        patched_outputs = cached_outputs.copy()\n        patched_outputs[frame_idx] = {}\n        patched_state["cached_frame_outputs"] = patched_outputs\n        return original(patched_state, frame_idx, refined_obj_id_to_mask)\n\n    build_output._first_point_output_fixed = True\n    model._build_sam2_output = build_output\n    return predictor\n\n\ndef clean_token(token):\n    """Remove surrounding whitespace; enter a token without embedded spaces or control characters."""\n    if token is None:\n        return None\n    if not isinstance(token, str):\n        raise ValueError("The Hugging Face token must be text.")\n    cleaned = token.strip()\n    if not cleaned or any(ord(character) <= 32 or ord(character) >= 127 for character in cleaned):\n        raise ValueError("Enter a nonempty Hugging Face token without embedded whitespace or non-ASCII characters.")\n    return cleaned\n\n\ndef load_predictor(max_objects=16, token=None, model_revision=None):\n    """Download approved weights using your token without storing it in the notebook."""\n    if isinstance(max_objects, bool) or not isinstance(max_objects, int) or not 1 <= max_objects <= 256:\n        raise ValueError("Choose an object limit between 1 and 256; larger limits require more memory.")\n    token = clean_token(token)\n    environment = check_environment()\n    from huggingface_hub import HfApi, hf_hub_download\n    from sam3.model_builder import build_sam3_multiplex_video_predictor\n\n    revision = HfApi().model_info(MODEL_REPOSITORY, revision=model_revision, token=token).sha\n    checkpoint = hf_hub_download(\n        MODEL_REPOSITORY, MODEL_FILENAME, revision=revision, token=token,\n    )\n    predictor = build_sam3_multiplex_video_predictor(\n        checkpoint_path=checkpoint, max_num_objects=max_objects,\n        use_fa3=False, compile=False, warm_up=False, async_loading_frames=False,\n    )\n    adapt_session_arguments(predictor)\n    adapt_point_output(predictor)\n    environment.update({"checkpoint_revision": revision, "checkpoint_file": Path(checkpoint).name,\n                        "max_objects": max_objects, "flash_attention_3": False, "compile": False})\n    return predictor, environment\n', 'ui.py': '"""Interactive annotation controls for a selected image tracking project."""\n\nfrom __future__ import annotations\n\nfrom html import escape\nimport math\n\nimport ipywidgets as widgets\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import Rectangle\nfrom matplotlib.widgets import RectangleSelector\nimport numpy as np\nfrom PIL import Image\n\nfrom .labels import draw_object_label\n\n\nclass TrackingUI:\n    """Display ``widget`` with an interactive Matplotlib backend in a notebook.\n\n    Coordinates use original image pixels. Contrast affects display only. Close\n    this UI before replacing its project; close the project separately.\n    """\n\n    def __init__(self, project):\n        self.project = project\n        self._pending = []\n        self._busy = False\n        self._closed = False\n        self._updating = False\n        self._displayed_frame = None\n        self._observers = []\n        self._buttons = []\n        count, self._height, self._width = project.sequence.shape\n        dtype = np.dtype(project.sequence.metadata["dtype"])\n        maximum = float(np.iinfo(dtype).max)\n        initial_low = float(project.prepared.get("low", 0))\n        initial_high = float(project.prepared.get("high", maximum))\n        self.frame = widgets.IntSlider(min=0, max=count - 1, value=0,\n                                       description="Frame", continuous_update=False)\n        self.frame_label = widgets.Label()\n        self.object = widgets.Dropdown(description="Object", options=[("New object", None)])\n        self.tool = widgets.ToggleButtons(\n            options=[("+ Point", "point_positive"), ("− Point", "point_negative"),\n                     ("+ Box", "box_positive"), ("− Box", "box_negative")],\n            value="point_positive", description="Tool")\n        saved = project.concept_prompt\n        self.concept = widgets.Text(value=saved["text"] if saved is not None else "",\n                                    description="Concept", placeholder="e.g. dog")\n        self.box_update = widgets.Dropdown(description="Box update",\n                                           options=[("Add to saved", "add"), ("Replace saved", "replace")])\n        self.saved_label = widgets.Label()\n        self.apply_concept = widgets.Button(description="Apply concept / boxes", button_style="primary")\n        self.apply_points = widgets.Button(description="Apply points", button_style="primary")\n        self.undo = widgets.Button(description="Undo pending")\n        self.clear = widgets.Button(description="Clear pending")\n        self.object_name = widgets.Text(description="Name", placeholder="Object name")\n        self.name_prefix = widgets.Text(value=project.name_prefix, description="Name prefix",\n                                        placeholder="Blank: concept or Object",\n                                        style={"description_width": "initial"})\n        self.apply_prefix = widgets.Button(description="Apply prefix to all")\n        self._displayed_object_name = ""\n        self.save_metadata = widgets.Button(description="Save name")\n        self.exclude = widgets.Button(description="Exclude object", button_style="warning")\n        self.direction = widgets.Dropdown(description="Direction", options=["both", "forward", "backward"])\n        self.track = widgets.Button(description="Track", button_style="success", icon="play")\n        self.progress = widgets.IntProgress(min=0, max=count, value=0, description="Tracked")\n        self.contrast = widgets.FloatRangeSlider(\n            min=0, max=maximum, value=(max(0, min(maximum, initial_low)),\n                                      max(0, min(maximum, initial_high))),\n            step=1, description="Display range", continuous_update=False,\n            readout_format=".0f", style={"description_width": "initial"})\n        self.alpha = widgets.FloatSlider(min=0, max=1, step=0.05, value=0.4,\n                                          description="Mask opacity", continuous_update=False,\n                                          style={"description_width": "initial"})\n        self.status = widgets.Label(value="Choose a tool and annotate the image.")\n        self.state_label = widgets.Label()\n        self.pending_label = widgets.Label()\n        self.legend = widgets.HTML()\n        with plt.ioff():\n            self.figure, self.axes = plt.subplots(figsize=(9, 6), constrained_layout=True)\n        canvas = self.figure.canvas\n        self._plot_is_widget = isinstance(canvas, widgets.Widget)\n        if self._plot_is_widget:\n            self._plot_widget = canvas\n            if hasattr(canvas, "toolbar_visible"):\n                canvas.toolbar_visible = True\n            if hasattr(canvas, "header_visible"):\n                canvas.header_visible = False\n            if hasattr(canvas, "resizable"):\n                canvas.resizable = True\n        else:\n            self._plot_widget = widgets.Output()\n        self._selector = None\n        self._click_connection = canvas.mpl_connect("button_press_event", self._on_click)\n        self._controls = [self.frame, self.object, self.tool, self.concept, self.box_update, self.apply_concept,\n                          self.apply_points, self.undo, self.clear, self.object_name,\n                          self.save_metadata, self.name_prefix, self.apply_prefix, self.exclude, self.direction,\n                          self.track, self.contrast, self.alpha]\n        self._mutations = [self.tool, self.concept, self.box_update, self.apply_concept, self.apply_points,\n                           self.undo, self.clear, self.object_name,\n                           self.save_metadata, self.name_prefix, self.apply_prefix, self.exclude, self.track, self.direction]\n        self.widget = widgets.VBox([\n            widgets.HTML("<b>SAM 3.1 · Interactive tracking</b>"),\n            widgets.Label("Frames and source indices are zero-based. Use the toolbar to zoom or pan."),\n            self._row(self.frame, self.frame_label),\n            self._row(self.contrast, self.alpha), self._plot_widget,\n            self._row(self.object, self.tool), self.pending_label,\n            self._row(self.concept, self.box_update, self.apply_concept, self.apply_points),\n            self.saved_label,\n            widgets.Label("Saved boxes are dashed. Reapplying detection replaces objects/IDs, discards point "\n                          "corrections and previous tracking; redo Track before export."),\n            self._row(self.undo, self.clear),\n            self._row(self.name_prefix, self.apply_prefix),\n            widgets.Label("Names use prefix + object ID. Applying to all replaces individual names."),\n            self._row(self.object_name, self.save_metadata, self.exclude),\n            self.legend, self._row(self.direction, self.track, self.progress),\n            self.state_label, self.status,\n            widgets.Label("Boxes provide detection examples; points target the selected object. "\n                          "Pending input clears on frame, object, or point/box tool changes."),\n        ], layout=widgets.Layout(width="100%", max_width="1000px"))\n        self._observe(self.frame, self._on_frame)\n        self._observe(self.object, self._on_object)\n        self._observe(self.tool, self._on_tool)\n        self._observe(self.contrast, self._on_display)\n        self._observe(self.alpha, self._on_display)\n        for button, callback in [(self.apply_concept, self._apply_concept),\n                                 (self.apply_points, self._apply_points),\n                                 (self.undo, self._undo), (self.clear, self._clear),\n                                 (self.save_metadata, self._save_metadata),\n                                 (self.apply_prefix, self._apply_prefix),\n                                 (self.exclude, self._exclude), (self.track, self._track)]:\n            button.on_click(callback)\n            self._buttons.append((button, callback))\n        self.refresh()\n\n    @staticmethod\n    def _row(*children):\n        return widgets.Box(children, layout=widgets.Layout(\n            display="flex", flex_flow="row wrap", align_items="center", width="100%"))\n\n    def _observe(self, widget, callback):\n        widget.observe(callback, names="value")\n        self._observers.append((widget, callback))\n\n    def _message(self, text):\n        self.status.value = str(text).replace("\\n", " ")[:400]\n\n    def _can_annotate(self):\n        return not (self._closed or self._busy or self.project.rebuild_required)\n\n    def _navigation_active(self):\n        toolbar = getattr(self.figure.canvas, "toolbar", None)\n        return bool(getattr(toolbar, "mode", "") or self.axes.get_navigate_mode())\n\n    def _on_click(self, event):\n        if not self._can_annotate() or self._navigation_active():\n            return\n        if not self.tool.value.startswith("point") or event.inaxes is not self.axes or event.button != 1:\n            return\n        if not self._valid_point(event.xdata, event.ydata):\n            return\n        self._pending.append({"kind": "point", "coordinates": [float(event.xdata), float(event.ydata)],\n                              "label": int(self.tool.value.endswith("positive"))})\n        self._message("Point added. Apply points to submit the pending annotations.")\n        self._redraw_safely()\n\n    def _valid_point(self, x, y):\n        return x is not None and y is not None and math.isfinite(x) and math.isfinite(y) and (\n            0 <= x < self._width and 0 <= y < self._height)\n\n    def _on_rectangle(self, press, release):\n        if not self._can_annotate() or self._navigation_active() or not self.tool.value.startswith("box"):\n            return\n        values = (press.xdata, press.ydata, release.xdata, release.ydata)\n        if any(value is None or not math.isfinite(value) for value in values):\n            return\n        if any(not 0 <= value <= bound for value, bound in zip(\n                values, (self._width, self._height, self._width, self._height))):\n            return\n        left, right = sorted((float(press.xdata), float(release.xdata)))\n        top, bottom = sorted((float(press.ydata), float(release.ydata)))\n        if right <= left or bottom <= top:\n            return\n        self._pending.append({"kind": "box", "coordinates": [left / self._width, top / self._height,\n                              (right - left) / self._width, (bottom - top) / self._height],\n                              "label": int(self.tool.value.endswith("positive"))})\n        self._message("Box added. Apply concept / boxes to update the saved examples and redetect.")\n        self._redraw_safely()\n\n    def _clear_pending(self, reason=None):\n        had_pending = bool(self._pending)\n        self._pending.clear()\n        if had_pending and reason:\n            self._message(f"Pending annotations cleared: {reason}.")\n\n    def _on_frame(self, change):\n        if self._updating or self._closed or self._busy:\n            return\n        self._clear_pending("frame changed")\n        self.refresh(sync_prefix=False)\n\n    def _on_object(self, change):\n        if self._updating or self._closed or self._busy:\n            return\n        self._clear_pending("object changed")\n        self.refresh(sync_prefix=False)\n\n    def _on_tool(self, change):\n        if self._updating or self._closed or self._busy:\n            return\n        if change["old"].split("_")[0] != change["new"].split("_")[0]:\n            self._clear_pending("point/box tool changed")\n        self.refresh(sync_prefix=False)\n\n    def _on_display(self, change):\n        if not (self._updating or self._closed or self._busy):\n            self._redraw_safely()\n\n    def _run(self, action, success):\n        if not self._can_annotate():\n            return\n        try:\n            action()\n        except Exception as error:\n            self._message(f"Error: {type(error).__name__}: {error}")\n        else:\n            self._message(success)\n        finally:\n            self.refresh(sync_prefix=False)\n\n    def _apply_concept(self, button=None):\n        def apply():\n            if any(entry["kind"] != "box" for entry in self._pending):\n                raise ValueError("Apply or clear pending points before applying a concept")\n            boxes = [entry for entry in self._pending if entry["kind"] == "box"]\n            self.project.update_concept(self.frame.value, self.concept.value,\n                                        boxes=[entry["coordinates"] for entry in boxes],\n                                        box_labels=[entry["label"] for entry in boxes],\n                                        mode=self.box_update.value, name_prefix=self.name_prefix.value)\n            self._clear_pending()\n            self.object.value = None\n            self.object_name.value = ""\n        self._run(apply, "Detection applied. Review the new objects and Track before exporting.")\n\n    def _apply_points(self, button=None):\n        def apply():\n            if any(entry["kind"] != "point" for entry in self._pending):\n                raise ValueError("Apply or clear pending boxes before applying points")\n            points = [entry for entry in self._pending if entry["kind"] == "point"]\n            previous_ids = set(self.project.objects)\n            selected = self.object.value\n            self._save_prefix()\n            name = self.object_name.value if selected is None or self.object_name.value != self._displayed_object_name else ""\n            self.project.add_points(self.frame.value, selected,\n                                    [entry["coordinates"] for entry in points],\n                                    [entry["label"] for entry in points], name=name)\n            self._clear_pending()\n            if selected is None:\n                added = sorted(set(self.project.objects) - previous_ids)\n                if added:\n                    self._sync_objects(added[0])\n        self._run(apply, "Points applied. Review masks and track when ready.")\n\n    def _undo(self, button=None):\n        if self._can_annotate() and self._pending:\n            self._pending.pop()\n            self._message("Last pending annotation removed.")\n            self._redraw_safely()\n\n    def _clear(self, button=None):\n        if self._can_annotate():\n            self._clear_pending()\n            self._message("Pending annotations cleared.")\n            self._redraw_safely()\n\n    def _save_metadata(self, button=None):\n        def save():\n            if self.object.value is None:\n                raise ValueError("Select an existing object first")\n            self.project.rename_object(self.object.value, self.object_name.value)\n        self._run(save, "Object name saved.")\n\n    def _save_prefix(self):\n        prefix = self.name_prefix.value.strip()\n        if prefix != self.project.name_prefix:\n            self.project.set_name_prefix(prefix)\n\n    def _apply_prefix(self, button=None):\n        self._run(lambda: self.project.set_name_prefix(self.name_prefix.value, rename_existing=True),\n                  "Prefix applied to all active objects and future objects.")\n\n    def _exclude(self, button=None):\n        def exclude():\n            if self.object.value is None:\n                raise ValueError("Select an existing object first")\n            self.project.remove_object(self.object.value, frame=self.frame.value)\n            self._clear_pending()\n        self._run(exclude, "Object excluded. Track again to update the result snapshot.")\n\n    def _track(self, button=None):\n        if not self._can_annotate():\n            return\n        if self._pending:\n            self._message("Apply or clear pending annotations before tracking.")\n            return\n        self._busy = True\n        self.progress.value = 0\n        self.progress.bar_style = "info"\n        self._set_controls()\n        self._message("Tracking…")\n        try:\n            self._save_prefix()\n            self.project.propagate(start_frame=self.frame.value, direction=self.direction.value,\n                                   progress=self._update_progress)\n        except Exception as error:\n            self.progress.bar_style = "danger"\n            self._message(f"Error: {type(error).__name__}: {error}")\n        else:\n            self.progress.bar_style = "warning" if self.project.dirty else "success"\n            self._message("Tracking finished. " + (\n                "Partial coverage; run both directions for complete export." if self.project.dirty\n                else "All selected frames are ready for export."))\n        finally:\n            self._busy = False\n            self.refresh(sync_prefix=False)\n\n    def _update_progress(self, completed, total):\n        self.progress.max = max(1, int(total))\n        self.progress.value = min(int(completed), self.progress.max)\n        self.progress.send_state()\n        self.figure.canvas.flush_events()\n\n    def _sync_objects(self, selected=None):\n        previous = self.object.value if selected is None else selected\n        options = [("New object", None)] + [\n            (f"{object_id}: {item[\'name\']}", object_id)\n            for object_id, item in sorted(self.project.objects.items()) if not item["excluded"]]\n        self._updating = True\n        try:\n            self.object.options = options\n            self.object.value = previous if previous in [value for _, value in options] else None\n        finally:\n            self._updating = False\n        if self.object.value != previous:\n            self._clear_pending("selected object is unavailable")\n\n    def _set_controls(self):\n        for control in self._controls:\n            control.disabled = self._closed or self._busy\n        if self.project.rebuild_required:\n            for control in self._mutations:\n                control.disabled = True\n        if self.object.value is None:\n            self.save_metadata.disabled = True\n            self.exclude.disabled = True\n        if self._selector is not None:\n            self._selector.set_active(self._can_annotate() and self.tool.value.startswith("box"))\n\n    def refresh(self, *, sync_prefix=True):\n        """Read project state and redraw without changing inference intensity limits."""\n        if self._closed:\n            return\n        if sync_prefix:\n            self.name_prefix.value = self.project.name_prefix\n            saved = self.project.concept_prompt\n            if saved is not None:\n                self.concept.value = saved["text"]\n        self._sync_objects()\n        item = self.project.objects.get(self.object.value)\n        if item is not None:\n            self.object_name.value = item["name"]\n        else:\n            self.object_name.value = ""\n        self._displayed_object_name = self.object_name.value\n        source_index = self.project.sequence.frame_indices[self.frame.value]\n        self.frame_label.value = f"Local {self.frame.value} / {self.frame.max} · source {source_index}"\n        saved = self.project.concept_prompt\n        if saved is None:\n            self.saved_label.value = "Saved examples: 0 · no reference frame"\n        else:\n            positive = sum(saved["box_labels"])\n            self.saved_label.value = (f"Saved examples: {len(saved[\'boxes\'])} ({positive} target / "\n                                      f"{len(saved[\'boxes\']) - positive} background) · reference local frame {saved[\'frame\']}")\n        if self.project.rebuild_required:\n            self.state_label.value = "Session failed: save successful events and replay in a new project."\n        elif self.project.dirty:\n            self.state_label.value = "Results need full tracking. Current-frame masks are prompt / partial previews."\n        else:\n            self.state_label.value = "Complete result snapshot: all selected frames tracked."\n        entries = []\n        for object_id, object_data in sorted(self.project.objects.items()):\n            color = self._color(object_data)\n            rgb = ",".join(str(round(channel * 255)) for channel in color)\n            selected = " <b>(selected)</b>" if self.object.value == object_id else ""\n            excluded = " (excluded)" if object_data["excluded"] else ""\n            entries.append(f\'<span style="color:rgb({rgb})">●</span> \'\n                           f\'{object_id}: {escape(str(object_data["name"]))}{selected}{excluded}\')\n        self.legend.value = "<br>".join(entries) or "No objects yet."\n        self._set_controls()\n        self._redraw_safely()\n\n    @staticmethod\n    def _color(item):\n        return np.clip(np.asarray(item["color"], dtype=float) / 255, 0, 1)\n\n    def _visible_masks(self):\n        if self.project.rebuild_required:\n            return {}\n        if self.project.current_frame_index == self.frame.value:\n            return self.project.current_frame_outputs or {}\n        if not self.project.dirty:\n            return self.project.results.get(self.frame.value) or {}\n        return {}\n\n    def _redraw_safely(self):\n        try:\n            self._draw()\n        except Exception as error:\n            self._displayed_frame = None\n            self.axes.clear()\n            self.figure.canvas.draw_idle()\n            self._message(f"Display error: {type(error).__name__}: {error}")\n\n    def _draw(self):\n        same_frame = self._displayed_frame == self.frame.value\n        previous_xlim, previous_ylim = self.axes.get_xlim(), self.axes.get_ylim()\n        if self._selector is not None:\n            self._selector.disconnect_events()\n        self.axes.clear()\n        self.axes.set_xlabel("X (pixels)")\n        self.axes.set_ylabel("Y (pixels)")\n        source_index = self.project.sequence.frame_indices[self.frame.value]\n        low, high = self.contrast.value\n        frame_image = self.project.sequence.preview(source_index, low=low, high=high)\n        extent = (-0.5, self._width - 0.5, self._height - 0.5, -0.5)\n        if frame_image is not None:\n            self.axes.imshow(frame_image, origin="upper", extent=extent, interpolation="nearest")\n            labels = Image.new("RGBA", (self._width, self._height))\n            for object_id, mask in self._visible_masks().items():\n                item = self.project.objects.get(object_id)\n                if item is None or item["excluded"]:\n                    continue\n                mask = np.asarray(mask, dtype=bool)\n                if mask.shape != (self._height, self._width):\n                    raise ValueError("Mask dimensions do not match the image")\n                rgba = np.zeros((*mask.shape, 4), dtype=float)\n                rgba[mask, :3] = self._color(item)\n                rgba[mask, 3] = self.alpha.value\n                self.axes.imshow(rgba, origin="upper", extent=extent, interpolation="nearest")\n                selected = object_id == self.object.value\n                draw_object_label(labels, mask, object_id, item, selected=selected)\n                if selected and mask.any() and not mask.all():\n                    self.axes.contour(mask, levels=[0.5], colors=["white"], linewidths=1.2)\n            self.axes.imshow(np.asarray(labels), origin="upper", extent=extent, interpolation="nearest")\n        saved = self.project.concept_prompt\n        if saved is not None and saved["frame"] == self.frame.value:\n            for box, label in zip(saved["boxes"], saved["box_labels"]):\n                left, top, width, height = box\n                self.axes.add_patch(Rectangle((left * self._width, top * self._height),\n                    width * self._width, height * self._height, fill=False,\n                    edgecolor="#39e68c" if label else "#ff5e72", linewidth=2, linestyle="dashed"))\n        for entry in self._pending:\n            positive = bool(entry["label"])\n            color = "#39e68c" if positive else "#ff5e72"\n            coordinates = entry["coordinates"]\n            if entry["kind"] == "point":\n                self.axes.plot(*coordinates, marker="+" if positive else "x", color=color,\n                               markersize=12, markeredgewidth=2, linestyle="none")\n            else:\n                left, top, width, height = coordinates\n                self.axes.add_patch(Rectangle((left * self._width, top * self._height),\n                    width * self._width, height * self._height, fill=False, edgecolor=color,\n                    linewidth=2, linestyle="solid" if positive else "dotted"))\n        self.axes.set_xlim(previous_xlim if same_frame else (-0.5, self._width - 0.5))\n        self.axes.set_ylim(previous_ylim if same_frame else (self._height - 0.5, -0.5))\n        self.axes.set_title(f"Local frame {self.frame.value} · source {source_index}")\n        points = sum(entry["kind"] == "point" for entry in self._pending)\n        self.pending_label.value = f"Pending: {points} points · {len(self._pending) - points} boxes"\n        self._selector = RectangleSelector(self.axes, self._on_rectangle, useblit=False,\n                                            button=[1], minspanx=0, minspany=0,\n                                            spancoords="data", interactive=False)\n        self._selector.set_active(self._can_annotate() and self.tool.value.startswith("box"))\n        self.figure.canvas.draw_idle()\n        self._displayed_frame = self.frame.value\n        if not self._plot_is_widget:\n            from IPython.display import clear_output, display\n            with self._plot_widget:\n                clear_output(wait=True)\n                display(self.figure)\n\n    def close(self):\n        """Disconnect annotation callbacks and close the figure, leaving the project open."""\n        if self._closed:\n            return\n        self._closed = True\n        for widget, callback in self._observers:\n            widget.unobserve(callback, names="value")\n        for button, callback in self._buttons:\n            button.on_click(callback, remove=True)\n        self.figure.canvas.mpl_disconnect(self._click_connection)\n        if self._selector is not None:\n            self._selector.disconnect_events()\n        self._set_controls()\n        plt.close(self.figure)\n'}
for filename, source in helper_sources.items():
    (package_dir / filename).write_text(source, encoding="utf-8")
if str(tool_root) not in sys.path:
    sys.path.insert(0, str(tool_root))
from sam31_colab.io import ImageSettings, TiffSequence
from sam31_colab.runtime import check_environment, load_predictor
from sam31_colab.engine import TrackingProject
from sam31_colab.ui import TrackingUI
from sam31_colab.colab_ui import ColabTrackingUI
print(check_environment())

## 3. Connect your images

Use a TIFF/OME-TIFF file, a grayscale or RGB JPG/JPEG or PNG file, or a folder of supported
images in natural filename order (for example, `frame2.png` before `frame10.png`).
A folder's files are concatenated in time. Each JPG/PNG file represents one frame.
All files must have matching XY dimensions, grayscale/RGB mode, and bit depth.
Use the file browser to copy the path into `INPUT_PATH` below.

Set `MOUNT_DRIVE=True` to access Google Drive. To upload files from your computer,
set `UPLOAD_FILES=True`; the upload directory will be printed. Upload only images belonging
to the same sequence. Large uploads use runtime disk space and may be lost on disconnect.

In [ ]:
MOUNT_DRIVE = False
UPLOAD_FILES = False

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if UPLOAD_FILES:
    from google.colab import files
    from uuid import uuid4
    upload_dir = Path("/content/sam31_inputs") / uuid4().hex[:12]
    upload_dir.mkdir(parents=True, exist_ok=False)
    uploaded = files.upload(target_dir=str(upload_dir))
    print("Input folder:", upload_dir)
    del uploaded

## 4. Choose the input and time range

Indices are **zero-based**; `STOP` is exclusive. For JPG/PNG, leave `AXES` empty (or use
`YX`) and `SERIES=0`, including RGB images. Palette, transparent, and animated JPG/PNG
inputs are unsupported; save a plain grayscale or RGB image before uploading.
JPEG compression can affect boundaries.
For TIFF, leave `AXES` empty to use trustworthy TIFF
metadata. If the reader reports an unknown page axis such as `QYX`, set `AXES="TYX"`
only after confirming that the pages represent time. Grayscale images normally use `YX`
and grayscale time-lapse sequences use `TYX`. For RGB TIFF, S means the three stored color
samples: common orders are `YXS` / `TYXS` or planar `SYX` / `TSYX`. For example, change
an unknown `QYXS` page axis to `TYXS` only if those pages represent time. Keep S in its
stored position. RGB input must be tagged as RGB in the TIFF, with exactly three samples;
RGBA and palette images are not supported.

T means time; tracking follows T only. Use single-channel grayscale or RGB input.
Non-RGB multichannel images (C > 1) and images with multiple Z slices are rejected.
Do not label channels or depth slices as time frames. Singleton C or Z axes in TIFF
metadata are accepted. RGB samples are processed together as one color image.
A selected time stride is recorded in the output; skipping frames can make tracking harder.

For grayscale, leave the intensity limits as `None` for one shared range estimated from
sampled frames (1st to 99.8th percentile), or enter fixed raw pixel values. For RGB, the
default range is 0-255 for uint8 (unchanged colors) or 0-65535 for uint16 (converted to
8-bit RGB). Any custom limits apply equally to R, G, and B for all frames; color samples
are never normalized independently. There is no automatic frame-by-frame normalization.
Empty or uniformly bright inputs may need explicit limits.

In [ ]:
INPUT_PATH = ""  # Enter an image file path or a folder of images from one sequence.
AXES = ""
SERIES = 0
START = 0
STOP = None
STEP = 1
INTENSITY_LOW = None
INTENSITY_HIGH = None

if not INPUT_PATH.strip():
    raise ValueError("Enter INPUT_PATH before running this cell.")
settings = ImageSettings(
    input_path=INPUT_PATH, axes=AXES, series=SERIES,
    start=START, stop=STOP, step=STEP,
    intensity_low=INTENSITY_LOW, intensity_high=INTENSITY_HIGH,
)
sequence = TiffSequence(settings)
print("Selected shape (T, Y, X):", sequence.shape)
print(sequence.metadata)

## 5. Prepare and inspect the model input

This cell creates a new PNG cache with the original XY dimensions. It does not change
the source images. Review the frame order, contrast, and histogram before
continuing. Reduce the selected frame range if runtime RAM or disk is insufficient.
The runtime holds model frames and output masks; long sequences are not split automatically.

In [ ]:
from uuid import uuid4
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

CACHE_ROOT = "/content/sam31_cache"
prepared = sequence.prepare(Path(CACHE_ROOT) / uuid4().hex)
first_source_frame = sequence.frame_indices[0]
preview = sequence.preview(first_source_frame, prepared["low"], prepared["high"])
figure, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(preview)
axes[0].set_title(f"Model input: source frame {first_source_frame}")
axes[0].set_xlabel("X (pixels)")
axes[0].set_ylabel("Y (pixels)")
raw_first_frame = sequence.raw_frame(first_source_frame)
sample_stride = max(1, preview.shape[0] * preview.shape[1] // 200000)
if sequence.is_rgb:
    for sample_index, sample_name, sample_color in ((0, "Red", "red"), (1, "Green", "green"), (2, "Blue", "blue")):
        axes[1].hist(raw_first_frame[..., sample_index].ravel()[::sample_stride], bins=100,
                     histtype="step", color=sample_color, label=sample_name)
else:
    axes[1].hist(raw_first_frame.ravel()[::sample_stride], bins=100)
axes[1].axvline(prepared["low"], color="red", label="Lower limit")
axes[1].axvline(prepared["high"], color="green", label="Upper limit")
axes[1].set_xlabel("Raw intensity")
axes[1].set_ylabel("Sampled pixel count")
axes[1].legend()
figure.tight_layout()
plt.show()
print("Shared intensity range:", prepared["low"], "to", prepared["high"])
print("Selected source frames:", list(sequence.frame_indices[:20]))
print("Approximate mask RAM per tracked object (GiB):", round(int(np.prod(sequence.shape)) / 2**30, 3))

## 6. Load SAM 3.1

Enter a read-only Hugging Face token in the hidden prompt each time you run this entire
cell. Pasted characters are not displayed; press Enter after pasting. Colab Secrets are
not read by this cell. The token is not included in project files. Model access must be approved
for your account. Surrounding whitespace is removed automatically; embedded whitespace
is rejected before connecting. Never print your token or share an error containing it.
Set `MAX_OBJECTS` to an upper bound for the entire recording, including
objects appearing later. Increasing it may require more GPU memory.

The notebook uses SAM 3.1 weights for both images and time series. FlashAttention 3 and
compilation are disabled for the initial run. Checkpoint revision and environment versions
are saved with results. No SAM2/SAM3 fallback is used.

In [ ]:
import getpass
from importlib.metadata import version, PackageNotFoundError

MAX_OBJECTS = 16
MODEL_REVISION = None  # Paste a saved checkpoint revision to reproduce a previous run.
hf_token = None
try:
    hf_token = getpass.getpass("Paste your Hugging Face read token, then press Enter: ").strip()
    if not hf_token or any(ord(character) <= 32 or ord(character) >= 127 for character in hf_token):
        raise ValueError("The token is empty or contains invalid characters. Run this entire cell again and enter the token.")
    predictor, environment = load_predictor(MAX_OBJECTS, token=hf_token, model_revision=MODEL_REVISION)
finally:
    hf_token = None
environment["packages"] = {}
for package in ("numpy", "tifffile", "Pillow", "ipywidgets", "ipympl", "torchvision", "sam3"):
    try:
        environment["packages"][package] = version(package)
    except PackageNotFoundError:
        environment["packages"][package] = "unavailable"
print("SAM 3.1 is ready.")
print(environment)

## 7. Open the annotation and tracking controls

Choose a frame and start with one of these approaches:

- Enter a short English concept, such as `dog`, `person`, or `car`, and detect candidates. Rectangles
  provide positive or negative visual examples for the concept; they are not guaranteed
  to identify only one object. Use one concept at a time.
- Select a new object and add foreground points. Add background points to exclude nearby
  regions. Apply the points, then select another object to add it.
- Select an existing object to refine its mask with points.

### Boxes versus points

**Boxes are visual examples for detecting matching objects throughout the image, including
outside the boxes.** They do not restrict detection to the enclosed object or region.
Positive boxes show what to find; negative boxes show unwanted examples. A negative box
does not simply erase the pixels inside it. Points, by contrast, select or refine one
object instance; clicking one object does not automatically detect all similar objects.

To detect multiple objects using **boxes alone**:

1. Start a fresh project with the cell below; save previous work before restarting it.
2. Select a representative frame and leave **Concept empty**.
3. Select **+ Box** and drag around one representative object, enclosing the whole object.
4. Optionally add more positive examples or use **− Box** around unwanted objects before applying.
5. Click **Apply concept / boxes**, not Apply points. Review masks and IDs across the entire image.
6. Refine individual objects with points if needed, then choose **Direction = both** and click **Track**.

Text and boxes can be combined when they describe the same kind of object. Results may
contain multiple objects, one object, or none. Set a sufficient **MAX_OBJECTS** in section 6.
### Update examples and redetect

Use **Box update** to choose how the next **Apply concept / boxes** uses your examples:

- **Add to saved**: combine saved examples with pending boxes on the same reference frame.
- **Replace saved**: use only the pending boxes. Redraw the full set to change examples;
  you may choose a different reference frame. To remove all boxes, keep Concept nonempty
  and apply Replace saved without pending boxes.

Saved examples are dashed on their reference frame; their count and reference frame are
shown in the panel. Undo pending and Clear pending affect only unapplied annotations.
You can edit Concept before reapplying, or switch from point-only input to text/boxes.

**Reapplying detection replaces active objects with new IDs and automatic names, clears
previous tracking results, and does not transfer individual names, exclusions, or point
corrections. Name prefix is retained.** Save the current analysis before redetecting if
you want to keep it. Review the new masks, add corrections, and run Track again before export.
No model reload or section 7 restart is needed to update examples.

In Colab, all controls use native HTML inputs and the image uses an HTML canvas.
Click to add points and drag to draw boxes. Adjust Zoom and scroll inside the image to
inspect a region. Undo removes the most recent pending point or box. Display contrast
changes only the preview; use Update display to apply the display limits and opacity.
Text detections are named automatically using the concept and object ID, such as `car 1`.
Set **Name prefix** before applying points or detecting objects to use another shared name;
the prefix persists when selecting New object. Leave it blank to use the concept, or Object
when no concept is set. **Apply prefix to all** renames all active objects and sets the prefix
for future objects, without rerunning inference. Individual names are still editable:
select an object, enter its exact name in **Name**, and click **Save name**. Names appear
beside the ID in the image and exported overlays, and in that object's mask folder name.
Default names keep ID-only image labels and folders. Renaming alone does not require
tracking again. Export to another empty or new folder to save changed names.
Non-Latin image labels require an installed font supporting those characters; use English
or alphanumeric names if a suitable font is unavailable. JSON retains original names.
Add newly appearing objects with new IDs when needed.

Track in **both** directions to cover the selected sequence. A forward run from frame 0
or a backward run from the final frame also covers it. Partial coverage cannot be exported
as a complete result. Add correction points on problem frames and track again.

After tracking finishes, use the **Frame** slider or **Frame number** to inspect any selected frame with
colored masks and object IDs. Adjust **Mask opacity** and **Display range** to inspect
boundaries. Frame numbers refer to the selected sequence; the source index is also shown.
Frames are displayed individually, not as a regularly spaced gallery. During tracking,
the status reads Tracking; frame navigation is available again when tracking finishes.
If a panel is displayed again, use the most recent panel; earlier panels are disconnected.

To start over entirely, save your current project first, then rerun the next cell.
Re-running it discards the live session, not saved exports. To update examples or text
within the current project, use Box update and Apply concept / boxes instead.

In [ ]:
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True
if "annotation_ui" in globals():
    annotation_ui.close()
if "project" in globals():
    project.close()
project = TrackingProject(sequence, predictor, prepared, environment=environment, max_objects=MAX_OBJECTS)
if in_colab:
    annotation_ui = ColabTrackingUI(project)
    annotation_ui.show()
else:
    from IPython import get_ipython
    get_ipython().run_line_magic("matplotlib", "widget")
    annotation_ui = TrackingUI(project)
    display(annotation_ui.widget)

## 8. Save prompts or reopen a saved project

Save the project JSON before disconnecting. Computing source fingerprints can take time for
large images or sequences. It records prompts and source fingerprints,
not GPU memory. Choose a new filename; existing files are not overwritten.
To reopen, select the same input/settings above, create a fresh project in section 7,
and set `REPLAY_PROJECT` below. Replay may recompute saved tracking steps and take time.
Different masks or identities during replay stop the process to avoid editing a wrong object.
Move your JSON and results to Drive or download them before the runtime expires.

In [ ]:
SAVE_PROJECT = ""  # Enter the full output path, including a new .json filename; leave empty to skip saving.
REPLAY_PROJECT = ""  # Enter the full path to an existing saved .json file to replay in a fresh project; otherwise leave empty.
if SAVE_PROJECT and REPLAY_PROJECT:
    raise ValueError("Choose either save or replay in one run.")
if SAVE_PROJECT:
    project.save_project(SAVE_PROJECT)
    print("Saved project:", SAVE_PROJECT)
if REPLAY_PROJECT:
    project.replay_project(REPLAY_PROJECT)
    annotation_ui.refresh()
    print("Project replay complete. Review the masks and object IDs.")

## 9. Export and review

Export only after the full selected sequence has been tracked following your latest
correction. Choose an **empty or new** output folder outside the input directory. Set `OUTPUT_MODE` to:

- `"mask"`: 8-bit binary TIFFs (background 0, foreground 255), one frame per file in
  `masks/object_<ID>_<name>/` for custom names, or `masks/object_<ID>/` for default names.
  Folder names replace characters unsuitable for file paths; JSON retains the original name.
  Each object has its own folder, so overlapping masks are preserved.
  An absent object has an all-zero mask. Object IDs are in folder names, not pixel values.
- `"overlay"`: RGB PNGs in `overlays/`, with colored masks, object IDs, and custom names on the
  displayed grayscale or RGB image. These use 8 bits per color channel and are for visual
  review. At overlapping pixels the lowest object ID is shown.
- `"both"`: both image formats.

Image dimensions are unchanged. Frame filenames count from zero within the selected time
range. Alongside the images, `project.json` records prompts, object IDs and names, image
preprocessing settings, source fingerprints, model versions, tracking history, and export
settings. Its `export.source_frame_indices` list maps each local frame index to the original
sequence index. Keep the same source images to replay the project.

Only the selected segmentation images and `project.json` are written. The JSON is written
last; an export interrupted before it is saved is incomplete. Choose a fresh folder when
retrying. Review masks, missing objects, and ID switches interactively before exporting.

In [ ]:
OUTPUT_DIRECTORY = ""  # Enter the full path to an empty or new output directory, preferably on Drive.
OUTPUT_MODE = "mask"  # Choose "mask", "overlay", or "both".
if not OUTPUT_DIRECTORY.strip():
    raise ValueError("Enter OUTPUT_DIRECTORY before exporting.")
result_directory = project.export(OUTPUT_DIRECTORY, output_mode=OUTPUT_MODE)
print("Results saved to:", result_directory)

## Checks before a larger analysis

1. Confirm T (time) and frame order against your acquisition. Use grayscale or RGB 2D
   images or a 2D time-lapse sequence. Non-RGB multichannel images and Z stacks are
   not supported. For RGB, check that the displayed red, green, and blue colors are correct.
2. Check at least two objects on the first, middle, and final frames, including contact
   or disappearance events. Compare representative masks with manual annotations.
3. Add correction points on a middle frame and track again. Confirm that the correction
   is retained and that old results cannot be exported accidentally.
4. Open the exported images in Fiji and verify image size and frame order. Binary TIFFs
   should be 8-bit with values 0 and 255; their folder names identify objects. Check overlay
   alignment and IDs if exported.
5. Save and replay a short project before relying on replay for a long experiment.

If memory is exhausted, keep your saved project and restart the runtime. Select fewer
frames or objects explicitly; the notebook does not silently lower resolution or split
tracks across independent sessions. CPU tests do not establish GPU accuracy or capacity.

Sources: [SAM 3.1 release](https://github.com/facebookresearch/sam3/blob/main/RELEASE_SAM3p1.md),
[official example](https://github.com/facebookresearch/sam3/blob/main/examples/sam3.1_video_predictor_example.ipynb),
[reference SAM2 notebook](https://github.com/YusukeKimata-Moo/SAM2-segmentation).
Weights and upstream code retain Meta's license and model-access conditions.